# Bold Gel Ball Pricing Strategy — Consolidated Analysis

**Objective:** Brand building via loyal user growth (Trial / Repeat / Lapse): define each size's most effective price point by understanding shopper flow.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | **Analysis Brief** | Objective, definitions, parameters, analysis flow |
| 2 | **Setup & Data Pull** | Imports, parameters, SQL queries with caching |
| A | **Pane A — Total Shopper & ASP Landscape** | Market sizing, pricing trends, renewal impact |
| B | **Pane B — Trial Shopper** | Trial acquisition by size, ASP elasticity, price gap |
| C | **Pane C — Repeat Shopper** | Cohort funnel, size migration, ASP band analysis |
| D | **Pane D — Lapsed Shopper** | Lapse rate by size/ASP, post-lapse destination |
| E | **Pane E — Shopper Flow** | Sankey, ASP-annotated funnel, strategic map |
| F | **Pane F — Strategic Summary** | Synthesized findings & recommendations |

**Created:** 2026-02-19 | **Consolidated:** 2026-02-27


---
### 📏 Canonical Definitions

| Term | Definition | Window |
|------|-----------|--------|
| **Trial Shopper** | A shopper who purchases the sub-brand/category with **no purchase history of that sub-brand/category in the prior 12 months** (365 days). Rolling lookback per purchase event, NOT first-ever. | 365-day lookback |
| **Repeat Shopper** | A trial shopper who makes ≥1 subsequent purchase of the **same sub-brand** within **180 days** (6 months) after their trial event. | 180-day forward window |
| **Lapsed Shopper** | A trial shopper who makes **no subsequent purchase** of the same sub-brand within **180 days** (6 months) after their trial event. | 180-day forward window |
| **ASP** | `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)` — weighted average selling price. | Per transaction/aggregation |
| **ASP Band (50 JPY bin)** | `FLOOR(ASP / 50) * 50` — ASP floored to nearest 50 JPY. | — |

**Repeat + Lapse are mutually exclusive and exhaustive** within the trial cohort.

### Analysis Flow
```
Total Shoppers (ASP Landscape)
  └── Trial Shoppers (new-to-brand)
        ├── Repeat Shoppers (retained within 6 months)
        │     └── Size migration (entry size → repeat size)
        └── Lapsed Shoppers (no return within 6 months)
              └── Destination tracking (where did they go?)

At each stage: How does PRICE affect the shopper's behavior?
```

### Data Sources (IDPOS_REFERENCE.md)
- **Fact table:** `cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw`
- **Product dim:** `id_pos_ai_1.prod_dim_ext_vw`
- **Shopper dim:** `id_pos_ai_1.shopper_dim_generic_vw`
- **Partition keys:** `sales_period_group_end_date_part`, `data_provider_code_part`


---
# Section 2 — Setup, Parameters & Data Pull


In [1]:
# ═══════════════════════════════════════════════════════════════════════
# 2-1. Imports & Connection
# ═══════════════════════════════════════════════════════════════════════
import os, time, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from dotenv import load_dotenv
import databricks.sql as sql

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

# ── Japanese font setup ────────────────────────────────────────────────
def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

# ── Databricks credentials ──────────────────────────────────────────────
load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

# ── Query helpers ────────────────────────────────────────────────────────
def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

def execute_query_long(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN,
                     _retry_stop_after_attempts_duration=3600) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

# ── Output helpers ───────────────────────────────────────────────────────
DATA_DIR   = Path('data')
OUTPUT_DIR = Path('output')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

def strip_tz(df):
    df = df.copy()
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)
    for col in df.columns:
        if df[col].dtype == 'object':
            try: df[col] = df[col].astype(str)
            except: pass
    return df

def fmt_km(v):
    if v >= 1_000_000: return f'{v/1_000_000:.1f}M'
    if v >= 1_000:     return f'{v/1_000:.1f}K'
    return str(int(v))

print('✅ Setup complete')


✅ Japanese font: MS Gothic
✅ Credentials loaded
✅ Setup complete


In [48]:
# ═══════════════════════════════════════════════════════════════════════
# 2-2. ALL Parameters (single source of truth)
# ═══════════════════════════════════════════════════════════════════════

# ── Target brands (half-width katakana per IDPOS_REFERENCE.md) ────────
BOLD_GB   = 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ'
ARIEL_GB   = 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

# ── Time windows ──────────────────────────────────────────────────────
LOOKBACK_START = '2024-01-01'   # Start of LAG lookback window
ANALYSIS_START = '2025-01-01'   # Analysis period start
ANALYSIS_END   = '2026-01-31'   # Analysis period end
RENEWAL_MONTH  = '2025-05-01'   # Product renewal breakpoint

# ── Canonical shopper classification windows ──────────────────────────
TRIAL_LOOKBACK_DAYS       = 365  # Trial = no purchase in prior 365 days
REPEAT_LAPSE_WINDOW_DAYS  = 180  # Repeat/Lapse = 180 days post-trial
LATEST_COHORT_END         = '2025-07-31'  # Latest cohort with 6-month follow-up
LAPSE_CUTOFF_DATE         = '2025-07-31'  # Confirm lapse after this date

# ── Retailer codes (8 national retailers) ─────────────────────────────
RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size hierarchy (physical size: small → large) ─────────────────────
SIZE_ORDER     = ['本体通常', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾃﾗｼﾞｬﾝﾎﾞ']
EXCLUDED_SIZES = ["詰替超ｼﾞｬﾝﾎﾞ","詰替超ﾃﾗｼﾞｬﾝﾎﾞ","ｿﾉﾀ","詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ","詰替超特大","詰替通常","詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ","詰替特大"]

SIZE_ROLE = {
    '本体通常':          'Trial Entry',
    '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ': 'Intermission', 
    '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ':  'Loyalty', 
    '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ':  'Loyalty',
    '詰替ﾃﾗｼﾞｬﾝﾎﾞ':  'Loyalty'}

# Manual capacity mapping (grams) — update from Phase 0 inspection
BOLD_GB_CAPACITY_G = {
    '本体通常':          690,
    '詰替超特大':        850,
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ':  1260,
    '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ': 1520,
}




# ── Frequency thresholds ──────────────────────────────────────────────
MIN_BIN_OBS    = 2     # weekly obs per 50 JPY bin (NB02 elasticity)
MIN_OBS_FREQ   = 10    # store×week obs per heatmap cell (NB02 heatmap)
MIN_FREQ       = 30    # shoppers per ASP band for lapse rate (NB04)
MIN_WEEK_STORE = 5     # week×store coverage for ASP band (NB04)

# ── Visualization colors ──────────────────────────────────────────────
BRAND_COLOR = {BOLD_GB: '#1E90FF', ARIEL_GB: '#FF6347'}
BRAND_LABEL = {BOLD_GB: 'Bold Gel Ball', ARIEL_GB: 'Ariel Gel Ball'}

def order_and_filter_sizes(sizes_list):
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

EXCLUDED_SIZES_SQL = ', '.join(f"'{s}'" for s in EXCLUDED_SIZES)

print('📋 Parameters loaded:')
print(f'  Analysis window : {ANALYSIS_START} → {ANALYSIS_END}')
print(f'  Lookback start  : {LOOKBACK_START}')
print(f'  Renewal month   : {RENEWAL_MONTH}')
print(f'  Cohort end      : {LATEST_COHORT_END}')
print(f'  Lapse cutoff    : {LAPSE_CUTOFF_DATE}')
print(f'  Retailers       : {len(RETAILER_CODES)}')
print(f'  Size order      : {SIZE_ORDER}')


📋 Parameters loaded:
  Analysis window : 2025-01-01 → 2026-01-31
  Lookback start  : 2024-01-01
  Renewal month   : 2025-05-01
  Cohort end      : 2025-07-31
  Lapse cutoff    : 2025-07-31
  Retailers       : 8
  Size order      : ['本体通常', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾃﾗｼﾞｬﾝﾎﾞ']


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# 2-3. Smart Cache Layer
# ═══════════════════════════════════════════════════════════════════════
# Set to True after first successful run → all queries load from disk instantly.
# Set to False to force re-query from Databricks.

RELOAD_FROM_CACHE = True   # ← Toggle this!

_query_log = []

def cached_query(name: str, sql_str: str, force: bool = False, long: bool = False) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{name}.parquet'
    csv_path     = DATA_DIR / f'{name}.csv'

    # Check cache
    if RELOAD_FROM_CACHE and not force and parquet_path.exists():
        mod_time = datetime.fromtimestamp(parquet_path.stat().st_mtime)
        age_days = (datetime.now() - mod_time).days
        df = pd.read_parquet(parquet_path)
        status = f'📂 CACHE ({age_days}d old)'
        if age_days > 7:
            status += ' ⚠️ >7 days old'
        print(f'{status} | {name}: {len(df):,} rows | {parquet_path}')
        _query_log.append({'query': name, 'rows': len(df), 'seconds': 0, 'source': 'cache'})
        return df

    # Run query
    print(f'⏳ Querying Databricks: {name}...', flush=True)
    t0 = time.time()
    df = execute_query_long(sql_str) if long else execute_query(sql_str)
    elapsed = time.time() - t0

    # Save cache
    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)
    print(f'✅ {name}: {len(df):,} rows | {elapsed:.1f}s | saved to {parquet_path}')
    _query_log.append({'query': name, 'rows': len(df), 'seconds': round(elapsed, 1), 'source': 'databricks'})
    return df

print(f'Cache mode: {"RELOAD FROM CACHE" if RELOAD_FROM_CACHE else "QUERY DATABRICKS"}')
print(f'Cache dir : {DATA_DIR.resolve()}')


Cache mode: RELOAD FROM CACHE
Cache dir : C:\Users\sugimoto.k.1\OneDrive - Procter and Gamble\work\01_analysis\python\IDPOS_template - Copy\workspace\SUD_pricing_strategy_20260227\data


---
## Data Pull — 8 Active Queries (reduced from 12)

| # | Query Name | Purpose | Status |
|---|-----------|---------|--------|
| Q1 | `monthly_sales` | Monthly sales/ASP/shoppers by brand × size | ✅ Active |
| ~~Q2~~ | ~~`asp_dq_check`~~ | ~~ASP data quality~~ | 💤 Commented out |
| Q3 | `product_metadata` | Product attributes for both brands | ✅ Active |
| Q4 | `trial_cohort_full` | **Unified** trial→repeat/lapse at shopper level | ✅ Active (heaviest) |
| ~~Q5~~ | ~~`category_trial`~~ | ~~Category-level trial~~ | 💤 Commented out |
| Q6 | `asp_weekly_retailer` | Weekly × retailer × store count (absorbs Q11) | ✅ Active |
| Q7 | `asp_band_universe` | All shoppers per (size, ASP band) | ✅ Active |
| Q8 | `dual_brand_all_shoppers` | **All** shoppers + `is_lapsed` flag (absorbs Q10) | ✅ Active |
| Q9 | `dual_brand_active` | Active shoppers per size, both brands | ✅ Active |
| ~~Q10~~ | ~~`at_risk_asp`~~ | ~~At-risk shoppers~~ | ♻️ Derived from Q8 |
| ~~Q11~~ | ~~`week_store_coverage`~~ | ~~Week×store count~~ | ♻️ Derived from Q6 |
| Q12 | `dual_brand_destination` | Post-lapse destination for both brands | ✅ Active |


In [4]:
# ── Q1: Monthly Sales Aggregation (replaces NB00+NB01+NB02 monthly) ──
q1_sql = f"""
SELECT
    DATE_TRUNC('month', CAST(idpos.sales_period_group_end_date_part AS DATE)) AS month,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    COUNT(DISTINCT idpos.shopper_key)   AS shoppers,
    SUM(idpos.pos_unit_sales_qty)       AS total_units,
    SUM(idpos.pos_sales_amt)            AS total_sales,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""
df_monthly = cached_query('monthly_sales', q1_sql)
df_monthly['month'] = pd.to_datetime(df_monthly['month'])
for col in ['shoppers', 'total_units', 'total_sales', 'weighted_asp']:
    df_monthly[col] = pd.to_numeric(df_monthly[col])


⏳ Querying Databricks: monthly_sales...
✅ monthly_sales: 205 rows | 677.7s | saved to data\monthly_sales.parquet


In [5]:
# # ── Q2 (commented out by user. pls comment out if later cell face an error due to this cell): ASP Data Quality Check ───────────────────────────────────────
# q2_sql = f"""
# SELECT
#     prod.jp_sub_brand_alter_lang_name                   AS sub_brand,
#     prod.jp_segment_4_name                              AS size_code,
#     COUNT(*)                                            AS total_rows,
#     SUM(CASE WHEN idpos.pos_sales_amt IS NULL THEN 1 ELSE 0 END)      AS null_sales,
#     SUM(CASE WHEN idpos.pos_unit_sales_qty IS NULL THEN 1 ELSE 0 END) AS null_units,
#     SUM(CASE WHEN idpos.pos_unit_sales_qty = 0 THEN 1 ELSE 0 END)    AS zero_units,
#     AVG(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS avg_asp,
#     PERCENTILE_APPROX(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0), 0.5) AS median_asp,
#     MIN(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS min_asp,
#     MAX(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS max_asp
# FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
# LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
#        ON idpos.prod_key = prod.prod_key
# LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
#        ON idpos.shopper_key = shopper.shopper_key
# WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
#   AND idpos.data_provider_code_part IN ({RETAILER_IN})
#   AND prod.jp_category_name = '{CATEGORY}'
#   AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
#   AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
#   AND shopper.member_ind = 'Y'
# GROUP BY 1, 2
# ORDER BY 1, 7 DESC
# """
# df_asp_dq = cached_query('asp_dq_check', q2_sql)
# for col in df_asp_dq.columns[2:]:
#     df_asp_dq[col] = pd.to_numeric(df_asp_dq[col])


In [6]:
# ── Q3: Product Metadata (replaces NB00 prod_family + capacity) ───────
q3_sql = f"""
SELECT DISTINCT
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_prod_family_1_name        AS prod_family,
    prod.jp_segment_4_name            AS size_code,
    prod.jp_prod_form_name            AS prod_form,
    prod.jp_prod_alter_lang_name      AS product_name,
    prod.jp_size_name                 AS size_name,
    prod.jp_pack_size_name            AS pack_size_name
FROM id_pos_ai_1.prod_dim_ext_vw prod
WHERE prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
ORDER BY 1, 2, 3
"""
df_product = cached_query('product_metadata', q3_sql)


⏳ Querying Databricks: product_metadata...
✅ product_metadata: 871 rows | 8.5s | saved to data\product_metadata.parquet


In [7]:
# ── Q4: Unified Trial → Repeat/Lapse Cohort (THE BIG ONE) ────────────
# Replaces NB02 subbrand_trial + NB03 cohort + NB05 journey
# Returns shopper-level trial→outcome for BOTH brands
q4_sql = f"""
WITH all_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
with_prev AS (
    SELECT *,
        LAG(purchase_date) OVER (
            PARTITION BY shopper_key, sub_brand ORDER BY purchase_date
        ) AS prev_purchase_date
    FROM all_purchases
),
trial_candidates AS (
    SELECT *
    FROM with_prev
    WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{LATEST_COHORT_END}'
      AND (prev_purchase_date IS NULL
           OR DATEDIFF(purchase_date, prev_purchase_date) > {TRIAL_LOOKBACK_DAYS})
),
trial_events AS (
    SELECT shopper_key, sub_brand, MIN(purchase_date) AS trial_date
    FROM trial_candidates
    GROUP BY 1, 2
),
trial_detail AS (
    SELECT te.shopper_key, te.sub_brand, te.trial_date,
           tc.size_code AS trial_size, tc.asp AS trial_asp
    FROM trial_events te
    INNER JOIN trial_candidates tc
           ON te.shopper_key = tc.shopper_key
          AND te.sub_brand   = tc.sub_brand
          AND te.trial_date  = tc.purchase_date
),
repeat_events AS (
    SELECT
        td.shopper_key, td.sub_brand, td.trial_date, td.trial_size, td.trial_asp,
        MIN(ap.purchase_date) AS repeat_date,
        MIN(ap.size_code)     AS repeat_size,
        MIN(ap.asp)           AS repeat_asp
    FROM trial_detail td
    LEFT JOIN all_purchases ap
           ON td.shopper_key = ap.shopper_key
          AND td.sub_brand   = ap.sub_brand
          AND ap.purchase_date > td.trial_date
          AND ap.purchase_date <= DATE_ADD(td.trial_date, {REPEAT_LAPSE_WINDOW_DAYS})
    GROUP BY 1, 2, 3, 4, 5
)
SELECT
    re.shopper_key, re.sub_brand, re.trial_date, re.trial_size, re.trial_asp,
    re.repeat_date, re.repeat_size, re.repeat_asp,
    CASE WHEN re.repeat_date IS NOT NULL THEN 'Repeat' ELSE 'Lapse' END AS outcome,
    DATEDIFF(re.repeat_date, re.trial_date) AS days_to_repeat
FROM repeat_events re
ORDER BY re.sub_brand, re.trial_date
"""
df_cohort = cached_query('trial_cohort_full', q4_sql, long=True)
df_cohort['trial_date']  = pd.to_datetime(df_cohort['trial_date'])
df_cohort['repeat_date'] = pd.to_datetime(df_cohort['repeat_date'])
for col in ['trial_asp', 'repeat_asp', 'days_to_repeat']:
    df_cohort[col] = pd.to_numeric(df_cohort[col])
print(f'  Bold: {len(df_cohort[df_cohort["sub_brand"]==BOLD_GB]):,} | '
      f'Ariel: {len(df_cohort[df_cohort["sub_brand"]==ARIEL_GB]):,}')


⏳ Querying Databricks: trial_cohort_full...
✅ trial_cohort_full: 1,135,036 rows | 733.3s | saved to data\trial_cohort_full.parquet
  Bold: 606,038 | Ariel: 528,998


In [8]:
# # ── Q5 (commented out by user. pls revise the query if later cell face an error due to this cell): Category Trial (category-level LAG) ──────────────────────────
# q5_sql = f"""
# WITH all_category_purchases AS (
#     SELECT
#         idpos.shopper_key,
#         prod.jp_sub_brand_alter_lang_name AS sub_brand,
#         prod.jp_segment_4_name            AS size_code,
#         CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
#     FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
#     LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
#     LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
#     WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
#       AND idpos.data_provider_code_part IN ({RETAILER_IN})
#       AND prod.jp_category_name = '{CATEGORY}'
#       AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
#       AND shopper.member_ind = 'Y'
#       AND idpos.pos_unit_sales_qty > 0
#     GROUP BY 1, 2, 3, 4
# ),
# with_prev_cat AS (
#     SELECT *, LAG(purchase_date) OVER (PARTITION BY shopper_key ORDER BY purchase_date) AS prev_cat_date
#     FROM all_category_purchases
# ),
# cat_trial_candidates AS (
#     SELECT * FROM with_prev_cat
#     WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
#       AND (prev_cat_date IS NULL OR DATEDIFF(purchase_date, prev_cat_date) > 365)
#       AND sub_brand IN ('{BOLD_GB}', '{ARIEL_GB}')
# ),
# cat_trial_events AS (
#     SELECT shopper_key, MIN(purchase_date) AS cat_trial_date FROM cat_trial_candidates GROUP BY 1
# ),
# cat_trial_detail AS (
#     SELECT cte.shopper_key, ctc.sub_brand, ctc.size_code, cte.cat_trial_date AS purchase_date
#     FROM cat_trial_events cte
#     INNER JOIN cat_trial_candidates ctc
#            ON cte.shopper_key = ctc.shopper_key AND cte.cat_trial_date = ctc.purchase_date
# )
# SELECT DATE_TRUNC('month', ctd.purchase_date) AS trial_month, ctd.sub_brand, ctd.size_code,
#        COUNT(DISTINCT ctd.shopper_key) AS category_trial_shoppers
# FROM cat_trial_detail ctd
# GROUP BY 1, 2, 3
# ORDER BY 1, 2, 3
# """
# df_cat_trial = cached_query('category_trial', q5_sql)
# df_cat_trial['trial_month'] = pd.to_datetime(df_cat_trial['trial_month'])
# df_cat_trial['category_trial_shoppers'] = pd.to_numeric(df_cat_trial['category_trial_shoppers'])


In [9]:
# ── Q6: Weekly Retailer-Level ASP + Units + Store Count (absorbs Q11) ─
# Added n_stores = COUNT(DISTINCT site_key) to derive week×store coverage
# in Python, eliminating the separate Q11 query.
q6_sql = f"""
SELECT
    CAST(idpos.sales_period_group_end_date_part AS DATE) AS week_end,
    idpos.data_provider_code_part AS retailer,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp,
    SUM(idpos.pos_unit_sales_qty) AS total_units,
    COUNT(DISTINCT idpos.shopper_key) AS buyer_count,
    COUNT(DISTINCT idpos.site_key) AS n_stores
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""
df_asp_weekly = cached_query('asp_weekly_retailer', q6_sql)
df_asp_weekly['week_end'] = pd.to_datetime(df_asp_weekly['week_end'])

for c in ['weighted_asp', 'total_units', 'buyer_count', 'n_stores']:
    df_asp_weekly[c] = pd.to_numeric(df_asp_weekly[c])

⏳ Querying Databricks: asp_weekly_retailer...
✅ asp_weekly_retailer: 28,040 rows | 137.2s | saved to data\asp_weekly_retailer.parquet


In [10]:
# ── Q7: ASP Band Universe (all Bold shoppers per size × ASP band) ───
q7_sql = f"""
WITH txn_asp AS (
    SELECT idpos.shopper_key, prod.jp_segment_4_name AS trial_size,
        CAST(FLOOR((SUM(idpos.pos_sales_amt)/SUM(idpos.pos_unit_sales_qty))/50)*50 AS BIGINT) AS asp_band
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{BOLD_GB}'
      AND prod.jp_segment_4_name NOT IN ({EXCLUDED_SIZES_SQL})
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY idpos.shopper_key, prod.jp_segment_4_name, idpos.sales_period_group_end_date_part
)
SELECT trial_size, asp_band, COUNT(DISTINCT shopper_key) AS all_shoppers
FROM txn_asp GROUP BY trial_size, asp_band
"""
df_universe = cached_query('asp_band_universe', q7_sql)
df_universe['asp_band']     = pd.to_numeric(df_universe['asp_band'])
df_universe['all_shoppers'] = pd.to_numeric(df_universe['all_shoppers'])


⏳ Querying Databricks: asp_band_universe...
✅ asp_band_universe: 244 rows | 125.9s | saved to data\asp_band_universe.parquet


In [11]:
# ── Q8: Dual-Brand ALL Shoppers + is_lapsed flag (absorbs Q10) ────────
# Returns ALL shoppers (active + lapsed) with their last purchase details.
# df_lapsed is derived in Python as a filter. Eliminates separate Q10 query.
q8_sql = f"""
WITH brand_purchases AS (
    SELECT idpos.shopper_key, prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt)/SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
last_purchase AS (
    SELECT shopper_key, sub_brand, MAX(purchase_date) AS last_date
    FROM brand_purchases GROUP BY 1, 2
    HAVING MAX(purchase_date) <= '{LAPSE_CUTOFF_DATE}'
),
lapse_check AS (
    SELECT lp.shopper_key, lp.sub_brand, lp.last_date,
        MAX(CASE WHEN bp.purchase_date > lp.last_date
                  AND bp.purchase_date <= DATE_ADD(lp.last_date, {REPEAT_LAPSE_WINDOW_DAYS})
                 THEN 1 ELSE 0 END) AS returned
    FROM last_purchase lp
    LEFT JOIN brand_purchases bp ON lp.shopper_key = bp.shopper_key AND lp.sub_brand = bp.sub_brand
        AND bp.purchase_date > lp.last_date
    GROUP BY 1, 2, 3
)
SELECT lc.shopper_key, lc.sub_brand, lc.last_date AS last_purchase_date,
       bp.size_code AS last_size, bp.asp AS last_asp,
       CASE WHEN lc.returned = 0 THEN 1 ELSE 0 END AS is_lapsed
FROM lapse_check lc
INNER JOIN brand_purchases bp ON lc.shopper_key = bp.shopper_key
    AND lc.sub_brand = bp.sub_brand AND lc.last_date = bp.purchase_date
"""
df_all_shoppers = cached_query('dual_brand_all_shoppers', q8_sql)
df_all_shoppers['last_purchase_date'] = pd.to_datetime(df_all_shoppers['last_purchase_date'])
df_all_shoppers['last_asp'] = pd.to_numeric(df_all_shoppers['last_asp'])
df_all_shoppers['is_lapsed'] = pd.to_numeric(df_all_shoppers['is_lapsed']).astype(int)

# Backward-compatible df_lapsed = only lapsed shoppers
df_lapsed = df_all_shoppers[df_all_shoppers['is_lapsed'] == 1].copy()
print(f'  All shoppers: {len(df_all_shoppers):,}')
print(f'  Bold lapsed: {len(df_lapsed[df_lapsed["sub_brand"]==BOLD_GB]):,}')
print(f'  Ariel lapsed: {len(df_lapsed[df_lapsed["sub_brand"]==ARIEL_GB]):,}')

⏳ Querying Databricks: dual_brand_all_shoppers...
✅ dual_brand_all_shoppers: 1,229,149 rows | 201.0s | saved to data\dual_brand_all_shoppers.parquet
  All shoppers: 1,229,149
  Bold lapsed: 635,923
  Ariel lapsed: 593,226


In [12]:
# ── Q9: Dual-Brand Active Shoppers per Size ───────────────────────────
q9_sql = f"""
SELECT prod.jp_sub_brand_alter_lang_name AS sub_brand, prod.jp_segment_4_name AS size_code,
       COUNT(DISTINCT idpos.shopper_key) AS active_shoppers
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{LAPSE_CUTOFF_DATE}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2
"""
df_active = cached_query('dual_brand_active', q9_sql)
df_active['active_shoppers'] = pd.to_numeric(df_active['active_shoppers'])


⏳ Querying Databricks: dual_brand_active...
✅ dual_brand_active: 18 rows | 91.2s | saved to data\dual_brand_active.parquet


In [13]:
# ── Q10: REMOVED — derived from Q8 (df_all_shoppers) ─────────────────
# Q10 was: all Bold shoppers with last ASP + is_lapsed flag.
# Now derived from expanded Q8 which returns ALL shoppers + is_lapsed.
df_at_risk = df_all_shoppers[df_all_shoppers['sub_brand'] == BOLD_GB][[
    'shopper_key', 'last_purchase_date', 'last_size', 'last_asp', 'is_lapsed'
]].rename(columns={'last_size': 'size_code', 'last_asp': 'asp'}).copy()
print(f'♻️ Q10 derived from Q8: {len(df_at_risk):,} Bold shoppers | {df_at_risk["is_lapsed"].sum():,} lapsed')


♻️ Q10 derived from Q8: 635,923 Bold shoppers | 635,923 lapsed


In [14]:
# ── Q11: REMOVED — derived from Q6 (df_asp_weekly + n_stores) ─────────
# Q11 was: (size, asp_band) → week_store_count.
# Now derived from Q6 which includes n_stores per (week, retailer, brand, size).
# We compute: for each (size, asp_band), sum n_stores across all weeks × retailers.
_q6_bold = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == BOLD_GB) &
    (df_asp_weekly['week_end'] <= pd.Timestamp(LAPSE_CUTOFF_DATE))
].copy()
_q6_bold['asp_band'] = (_q6_bold['weighted_asp'] // 50 * 50).astype(int)
df_week_store = _q6_bold.groupby(['size_code', 'asp_band']).agg(
    week_store_count=('n_stores', 'sum')
).reset_index()
df_week_store['asp_band'] = df_week_store['asp_band'].astype(int)
df_week_store['week_store_count'] = df_week_store['week_store_count'].astype(int)
print(f'♻️ Q11 derived from Q6: {len(df_week_store):,} (size × asp_band) combinations')


♻️ Q11 derived from Q6: 133 (size × asp_band) combinations


In [15]:
# ── Q12: Dual-Brand Post-Lapse Destination ────────────────────────────
q12_sql = f"""
WITH brand_last AS (
    SELECT idpos.shopper_key, prod.jp_sub_brand_alter_lang_name AS source_brand,
        MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS last_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2
    HAVING MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) <= DATE('{LAPSE_CUTOFF_DATE}')
),
next_laundry AS (
    SELECT bl.shopper_key, bl.source_brand,
        prod.jp_sub_brand_alter_lang_name AS next_sub_brand,
        prod.jp_segment_4_name            AS next_size,
        ROW_NUMBER() OVER (PARTITION BY bl.shopper_key, bl.source_brand
                           ORDER BY CAST(idpos.sales_period_group_end_date_part AS DATE)) AS rn
    FROM brand_last bl
    INNER JOIN cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
           ON bl.shopper_key = idpos.shopper_key
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    WHERE CAST(idpos.sales_period_group_end_date_part AS DATE) > bl.last_date
      AND idpos.sales_period_group_end_date_part <= '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND idpos.pos_unit_sales_qty > 0
)
SELECT shopper_key, source_brand, next_sub_brand, next_size
FROM next_laundry WHERE rn = 1
"""
df_destination = cached_query('dual_brand_destination', q12_sql, long=True)
print(f'  Bold→dest: {len(df_destination[df_destination["source_brand"]==BOLD_GB]):,}')
print(f'  Ariel→dest: {len(df_destination[df_destination["source_brand"]==ARIEL_GB]):,}')


⏳ Querying Databricks: dual_brand_destination...
✅ dual_brand_destination: 640,033 rows | 1032.6s | saved to data\dual_brand_destination.parquet
  Bold→dest: 325,821
  Ariel→dest: 314,212


In [16]:
# ── Data Pull Summary ─────────────────────────────────────────────────
print('\n' + '=' * 70)
print('DATA PULL SUMMARY')
print('=' * 70)
summary_df = pd.DataFrame(_query_log)
print(summary_df.to_string(index=False))
total_rows = summary_df['rows'].sum()
total_secs = summary_df['seconds'].sum()
cache_pct  = (summary_df['source'] == 'cache').sum() / len(summary_df) * 100
print(f'\nTotal: {total_rows:,} rows | {total_secs:.0f}s elapsed | {cache_pct:.0f}% from cache')



DATA PULL SUMMARY
                  query    rows  seconds     source
          monthly_sales     205    677.7 databricks
       product_metadata     871      8.5 databricks
      trial_cohort_full 1135036    733.3 databricks
    asp_weekly_retailer   28040    137.2 databricks
      asp_band_universe     244    125.9 databricks
dual_brand_all_shoppers 1229149    201.0 databricks
      dual_brand_active      18     91.2 databricks
 dual_brand_destination  640033  1,032.6 databricks

Total: 3,033,596 rows | 3007s elapsed | 0% from cache


---
# Pane A — Total Shopper & ASP Landscape
*Market sizing, pricing trends, and renewal impact across all sizes.*


In [17]:
# ── A-1: Monthly Sales Summary Table ──────────────────────────────────
df_asp = df_monthly[~df_monthly['size_code'].isin(EXCLUDED_SIZES)].copy()
renewal_date = pd.Timestamp(RENEWAL_MONTH)

# Summary table per brand × size
size_summary = df_asp.groupby(['sub_brand', 'size_code']).agg(
    total_shoppers=('shoppers', 'sum'),
    total_units=('total_units', 'sum'),
    total_sales=('total_sales', 'sum'),
).reset_index()
size_summary['weighted_asp'] = (size_summary['total_sales'] / size_summary['total_units']).round(1)
size_summary['sales_share_%'] = size_summary.groupby('sub_brand')['total_sales'].transform(
    lambda x: (x / x.sum() * 100).round(1)
)

print('📊 DATA TABLE: Size Summary by Brand')
print('=' * 90)
for brand in [BOLD_GB, ARIEL_GB]:
    print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
    b = size_summary[size_summary['sub_brand'] == brand].copy()
    b['size_code'] = pd.Categorical(b['size_code'],
        categories=order_and_filter_sizes(b['size_code']), ordered=True)
    b = b.sort_values('size_code')
    print(b[['size_code', 'total_shoppers', 'total_units', 'total_sales', 'weighted_asp', 'sales_share_%']].to_string(index=False))

# Save table
size_summary.to_csv(OUTPUT_DIR / 'pane_a_size_summary.csv', index=False)


📊 DATA TABLE: Size Summary by Brand

▶ Bold Gel Ball
    size_code  total_shoppers  total_units     total_sales  weighted_asp  sales_share_%
         本体通常          864883  1,549,928.0   391,028,566.0         252.3            8.1
        詰替超特大              36         54.0        22,167.0         410.5            0.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ               4          6.0         8,448.0       1,408.0            0.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ          413674    489,851.0   891,948,231.0       1,820.9           18.4
          NaN          397210    519,975.0 1,240,702,805.0       2,386.1           25.6
          NaN          195984    230,150.0   674,463,474.0       2,930.5           13.9
          NaN         1362482  1,819,825.0 1,656,843,474.0         910.4           34.1

▶ Ariel Gel Ball
   size_code  total_shoppers  total_units     total_sales  weighted_asp  sales_share_%
        本体通常          606503  1,127,080.0   275,965,584.0         244.9            5.5
       詰替超特大              32         43.0        20

In [18]:
# ── A-2: Combined Bold vs Ariel ASP Trend per Size ──────────────────
combined_data = df_asp.copy()
combined_sizes = order_and_filter_sizes(combined_data['size_code'].unique())

if combined_sizes:
    n_s = len(combined_sizes)
    n_c = 2
    n_r = (n_s + n_c - 1) // n_c

    fig_comb = make_subplots(rows=n_r, cols=n_c, subplot_titles=combined_sizes,
                             vertical_spacing=0.1, horizontal_spacing=0.1)

    for idx, size in enumerate(combined_sizes):
        r, c = idx // n_c + 1, idx % n_c + 1
        for brand, cfg in {BOLD_GB: ('#1E90FF', 'solid', 'Bold Gel Ball'),
                           ARIEL_GB: ('#FF6347', 'dot', 'Ariel Gel Ball')}.items():
            sub = combined_data[(combined_data['sub_brand']==brand) & (combined_data['size_code']==size)].sort_values('month')
            if len(sub) == 0: continue
            fig_comb.add_trace(go.Scatter(
                x=sub['month'], y=sub['weighted_asp'], mode='lines+markers',
                name=cfg[2], line=dict(color=cfg[0], dash=cfg[1]), marker=dict(size=5),
                showlegend=(idx==0)), row=r, col=c)
        fig_comb.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray', opacity=0.4, row=r, col=c)

    fig_comb.update_layout(height=320*n_r, title_text='Bold vs Ariel — Monthly ASP per Size',
                           template='plotly_white', legend=dict(orientation='h', yanchor='bottom', y=-0.08))
    fig_comb.update_yaxes(title_text='ASP (JPY)')
    fig_comb.show()

# ── DATA TABLE: Monthly ASP per Brand × Size ─────────────────────────
asp_table = combined_data[combined_data['size_code'].isin(combined_sizes)].pivot_table(
    index=['sub_brand', 'size_code'], columns='month', values='weighted_asp', aggfunc='mean'
).round(0)
print('\n📊 DATA TABLE: Monthly ASP — Bold vs Ariel per Size (JPY)')
print('=' * 120)
print(asp_table.to_string())
asp_table.reset_index().to_csv(OUTPUT_DIR / 'pane_a_asp_trend.csv', index=False)


📊 DATA TABLE: Monthly ASP — Bold vs Ariel per Size (JPY)
month                         2025-01-01 00:00:00+00:00  2025-02-01 00:00:00+00:00  2025-03-01 00:00:00+00:00  2025-04-01 00:00:00+00:00  2025-05-01 00:00:00+00:00  2025-06-01 00:00:00+00:00  2025-07-01 00:00:00+00:00  2025-08-01 00:00:00+00:00  2025-09-01 00:00:00+00:00  2025-10-01 00:00:00+00:00  2025-11-01 00:00:00+00:00  2025-12-01 00:00:00+00:00  2026-01-01 00:00:00+00:00
sub_brand      size_code                                                                                                                                                                                                                                                                                                                                                                   
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  本体通常                               321.0                      223.0                      191.0                      241.0                      302.0                   

In [19]:
# # ── A-3 (commented out by user): Pre vs Post Renewal Comparison ───────────────────────────────
# df_asp['period'] = df_asp['month'].apply(lambda x: 'Pre-Renewal' if x < renewal_date else 'Post-Renewal')

# comparison = df_asp.groupby(['sub_brand', 'size_code', 'period']).agg(
#     total_sales=('total_sales', 'sum'), total_units=('total_units', 'sum'),
#     avg_shoppers_per_month=('shoppers', 'mean'),
# ).reset_index()
# comparison['weighted_asp'] = comparison['total_sales'] / comparison['total_units']

# flat = comparison.pivot_table(index=['sub_brand', 'size_code'], columns='period',
#                               values='weighted_asp', aggfunc='first').reset_index()
# if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
#     flat['asp_change_%'] = ((flat['Post-Renewal'] - flat['Pre-Renewal']) / flat['Pre-Renewal'] * 100).round(1)
#     flat['asp_change_jpy'] = (flat['Post-Renewal'] - flat['Pre-Renewal']).round(0)

# print('📊 DATA TABLE: Pre vs Post Renewal ASP')
# print('=' * 80)
# for brand in [BOLD_GB, ARIEL_GB]:
#     print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
#     print(flat[flat['sub_brand'] == brand].to_string(index=False))

# # Grouped bar chart
# if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
#     plot_data = flat.copy()
#     plot_data['label'] = plot_data.apply(
#         lambda r: BRAND_LABEL.get(r['sub_brand'], r['sub_brand']) + ' | ' + r['size_code'], axis=1)
#     fig3 = go.Figure()
#     fig3.add_trace(go.Bar(name='Pre-Renewal', x=plot_data['label'], y=plot_data['Pre-Renewal'], marker_color='#6495ED'))
#     fig3.add_trace(go.Bar(name='Post-Renewal', x=plot_data['label'], y=plot_data['Post-Renewal'], marker_color='#FF6347'))
#     fig3.update_layout(barmode='group', title='ASP Before vs After Renewal', yaxis_title='Weighted ASP (JPY)',
#                        template='plotly_white', height=500)
#     fig3.show()

# flat.to_csv(OUTPUT_DIR / 'pane_a_pre_post_renewal.csv', index=False)


In [20]:
# ── A-4: Unit Ratio vs ASP Gap per Size ───────────────────────────────
_ug = df_asp.copy()
_plot_sizes = order_and_filter_sizes(_ug['size_code'].unique())
_units = _ug.groupby(['month', 'sub_brand', 'size_code'])['total_units'].sum().reset_index()
_asp_m = _ug.groupby(['month', 'sub_brand', 'size_code']).apply(
    lambda d: d['total_sales'].sum() / d['total_units'].sum()).reset_index(name='weighted_asp')

n_s = len(_plot_sizes); n_c = 2; n_r = (n_s + n_c - 1) // n_c
_specs = [[{"secondary_y": True}, {"secondary_y": True}] for _ in range(n_r)]
fig_ug = make_subplots(rows=n_r, cols=n_c, subplot_titles=_plot_sizes, specs=_specs,
                       vertical_spacing=0.14, horizontal_spacing=0.12)

ratio_gap_rows = []
for idx, size in enumerate(_plot_sizes):
    r, c = idx // n_c + 1, idx % n_c + 1
    _bold_u = _units[(_units['sub_brand']==BOLD_GB) & (_units['size_code']==size)][['month','total_units']].rename(columns={'total_units':'bold_units'})
    _ariel_u = _units[(_units['sub_brand']==ARIEL_GB) & (_units['size_code']==size)][['month','total_units']].rename(columns={'total_units':'ariel_units'})
    _ratio = pd.merge(_bold_u, _ariel_u, on='month', how='inner').sort_values('month')
    _ratio['unit_ratio'] = (_ratio['bold_units'] / _ratio['ariel_units'] * 100).round(1)

    if len(_ratio) > 0:
        fig_ug.add_trace(go.Scatter(x=_ratio['month'], y=_ratio['unit_ratio'], mode='lines+markers',
            name='Unit Ratio (%)', line=dict(color='#1E90FF', width=2.5),
            marker=dict(size=7, color=['#1E90FF' if v>=100 else '#FF6347' for v in _ratio['unit_ratio']]),
            showlegend=(idx==0), legendgroup='ratio'), row=r, col=c, secondary_y=False)
        fig_ug.add_hline(y=100, line_dash='dot', line_color='#888', opacity=0.7, row=r, col=c)

    _bold_a = _asp_m[(_asp_m['sub_brand']==BOLD_GB) & (_asp_m['size_code']==size)][['month','weighted_asp']].rename(columns={'weighted_asp':'bold_asp'})
    _ariel_a = _asp_m[(_asp_m['sub_brand']==ARIEL_GB) & (_asp_m['size_code']==size)][['month','weighted_asp']].rename(columns={'weighted_asp':'ariel_asp'})
    _gap = pd.merge(_bold_a, _ariel_a, on='month', how='inner').sort_values('month')
    _gap['asp_gap'] = (_gap['bold_asp'] - _gap['ariel_asp']).round(0)

    if len(_gap) > 0:
        fig_ug.add_trace(go.Bar(x=_gap['month'], y=_gap['asp_gap'], name='ASP Gap (¥)',
            marker_color=['#FFB347' if v>=0 else '#66CDAA' for v in _gap['asp_gap']], opacity=0.4,
            showlegend=(idx==0), legendgroup='gap'), row=r, col=c, secondary_y=True)

    fig_ug.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray', opacity=0.4, row=r, col=c)
    fig_ug.update_yaxes(title_text='Unit Ratio (%)', secondary_y=False, row=r, col=c)
    fig_ug.update_yaxes(title_text='ASP Gap (¥)', secondary_y=True, row=r, col=c)

    # Collect table data
    _merged = _ratio.merge(_gap[['month','bold_asp','ariel_asp','asp_gap']], on='month', how='outer')
    _merged['size_code'] = size
    ratio_gap_rows.append(_merged)

fig_ug.update_layout(height=380*n_r, title_text='Bold / Ariel Unit Ratio vs ASP Gap — by Size',
                     template='plotly_white', barmode='overlay',
                     legend=dict(orientation='h', yanchor='bottom', y=-0.06, x=0))
fig_ug.show()

# ── DATA TABLE: Unit Ratio & ASP Gap ─────────────────────────────────
if ratio_gap_rows:
    ratio_gap_df = pd.concat(ratio_gap_rows, ignore_index=True)
    ratio_gap_df = ratio_gap_df[['size_code','month','bold_units','ariel_units','unit_ratio','bold_asp','ariel_asp','asp_gap']]
    ratio_gap_df = ratio_gap_df.sort_values(['size_code','month'])
    print('\n📊 DATA TABLE: Unit Ratio & ASP Gap — by Size × Month')
    print('=' * 120)
    print(ratio_gap_df.to_string(index=False))
    ratio_gap_df.to_csv(OUTPUT_DIR / 'pane_a_ratio_gap.csv', index=False)


📊 DATA TABLE: Unit Ratio & ASP Gap — by Size × Month
  size_code                     month  bold_units  ariel_units  unit_ratio  bold_asp  ariel_asp  asp_gap
       本体通常 2025-01-01 00:00:00+00:00    62,280.0     35,488.0       175.5     309.7      320.7    -11.0
       本体通常 2025-02-01 00:00:00+00:00    63,717.0     84,778.0        75.2     290.3      223.0     67.0
       本体通常 2025-03-01 00:00:00+00:00    59,431.0    273,898.0        21.7     290.5      191.2     99.0
       本体通常 2025-04-01 00:00:00+00:00   168,165.0     90,287.0       186.3     215.7      241.5    -26.0
       本体通常 2025-05-01 00:00:00+00:00   275,563.0     44,372.0       621.0     202.4      302.0   -100.0
       本体通常 2025-06-01 00:00:00+00:00   138,581.0     48,629.0       285.0     243.8      323.1    -79.0
       本体通常 2025-07-01 00:00:00+00:00    83,125.0     43,135.0       192.7     301.1      353.4    -52.0
       本体通常 2025-08-01 00:00:00+00:00    66,272.0     48,254.0       137.3     339.9      335.4      4.0
 

In [21]:
# # ── A-5(commented out by user): Per-Dose ASP Comparison ──────────────────────────────────────
# bold_post = df_asp[(df_asp['sub_brand']==BOLD_GB) & (df_asp['month']>=renewal_date)]
# bold_asp_by_size = bold_post.groupby('size_code').agg(total_sales=('total_sales','sum'), total_units=('total_units','sum')).reset_index()
# bold_asp_by_size['weighted_asp'] = bold_asp_by_size['total_sales'] / bold_asp_by_size['total_units']
# bold_asp_by_size['capacity_g'] = bold_asp_by_size['size_code'].map(BOLD_GB_CAPACITY_G)
# bold_asp_by_size['asp_per_gram'] = (bold_asp_by_size['weighted_asp'] / bold_asp_by_size['capacity_g']).round(2)
# bold_asp_by_size['sub_brand'] = BOLD_GB

# ariel_post = df_asp[(df_asp['sub_brand']==ARIEL_GB) & (df_asp['month']>=renewal_date)]
# ariel_asp_by_size = ariel_post.groupby('size_code').agg(total_sales=('total_sales','sum'), total_units=('total_units','sum')).reset_index()
# ariel_asp_by_size['weighted_asp'] = ariel_asp_by_size['total_sales'] / ariel_asp_by_size['total_units']
# ariel_asp_by_size['capacity_g'] = np.nan
# ariel_asp_by_size['asp_per_gram'] = np.nan
# ariel_asp_by_size['sub_brand'] = ARIEL_GB

# dose_table = pd.concat([bold_asp_by_size, ariel_asp_by_size], ignore_index=True)
# print('📊 DATA TABLE: Per-Dose ASP (Post-Renewal)')
# print('=' * 80)
# print(dose_table[['sub_brand','size_code','weighted_asp','capacity_g','asp_per_gram']].to_string(index=False))
# dose_table.to_csv(OUTPUT_DIR / 'pane_a_dose_asp.csv', index=False)


---
# Pane B — Trial Shopper
*Trial acquisition by size, head-to-head comparison, ASP elasticity, and price gap analysis.*


In [22]:
# ── B-1: Derive monthly trial from unified cohort ─────────────────────
# Sub-brand trial per month per size (from Q4 cohort)
# Revert rename if it was already applied (idempotent)
if 'size_code' in df_cohort.columns and 'trial_size' not in df_cohort.columns:
    df_cohort.rename(columns={'size_code': 'trial_size'}, inplace=True)
df_cohort['trial_month'] = df_cohort['trial_date'].dt.to_period('M').dt.to_timestamp()
df_subbrand_trial = df_cohort.groupby(['trial_month', 'sub_brand', 'trial_size']).agg(
    subbrand_trial_shoppers=('shopper_key', 'nunique')
).reset_index().rename(columns={'trial_size': 'size_code'})

# Q5 (category trial) is commented out — use sub-brand trial only
df_trial = df_subbrand_trial.copy()

# Merge total monthly buyers (from Q1 monthly_sales — rename month)
_dm = df_monthly.copy()
_dm['month'] = pd.to_datetime(_dm['month']).dt.tz_localize(None)  # strip tz
df_monthly_buyers = _dm.rename(columns={'month': 'trial_month'}).groupby(
    ['trial_month', 'sub_brand', 'size_code'])['shoppers'].first().reset_index().rename(
    columns={'shoppers': 'total_shoppers'})
df_trial = df_trial.merge(df_monthly_buyers, on=['trial_month', 'sub_brand', 'size_code'], how='left')
df_trial['trial_rate'] = (df_trial['subbrand_trial_shoppers'] / df_trial['total_shoppers'].replace(0, np.nan) * 100).round(2)

# Data table
print('📊 DATA TABLE: Trial Summary by Brand × Size')
print('=' * 80)
summary = df_trial.groupby(['sub_brand', 'size_code']).agg(
    total_subbrand_trial=('subbrand_trial_shoppers', 'sum'),
).reset_index()
for brand in [BOLD_GB, ARIEL_GB]:
    print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
    print(summary[summary['sub_brand']==brand].to_string(index=False))

summary.to_csv(OUTPUT_DIR / 'pane_b_trial_summary.csv', index=False)

📊 DATA TABLE: Trial Summary by Brand × Size

▶ Bold Gel Ball
     sub_brand     size_code  total_subbrand_trial
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常                251172
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ         詰替超特大                    11
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                     2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ     詰替超ｼﾞｬﾝﾎﾞ                   114
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ                 65385
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          詰替通常                     2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ    詰替ﾃﾗｼﾞｬﾝﾎﾞ                 17431
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ                210077
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                 61643
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ           ｿﾉﾀ                   201

▶ Ariel Gel Ball
    sub_brand     size_code  total_subbrand_trial
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          本体通常                184676
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ         詰替超特大                    12
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ     詰替超ｼﾞｬﾝﾎﾞ                   231
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ                 63179
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ    詰替ﾃﾗｼﾞｬﾝﾎﾞ                 26325
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ                184798
ｱﾘｴｰﾙｼﾞｪﾙﾎ

In [23]:
# ── B-2: Combined Trial Trend — Bars + Trial Rate + Index ─────────────
combined_trial = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].copy()
comb_sizes = order_and_filter_sizes(combined_trial['size_code'].unique())

if comb_sizes:
    _piv = combined_trial.groupby(['trial_month', 'sub_brand', 'size_code']).agg(
        subbrand_trial=('subbrand_trial_shoppers', 'sum'), trial_rate=('trial_rate', 'mean')).reset_index()
    _a = _piv[_piv['sub_brand']==BOLD_GB][['trial_month','size_code','subbrand_trial','trial_rate']].rename(
        columns={'subbrand_trial':'ariel_trial','trial_rate':'ariel_rate'})
    _b = _piv[_piv['sub_brand']==ARIEL_GB][['trial_month','size_code','subbrand_trial']].rename(
        columns={'subbrand_trial':'attack_trial'})
    _idx_df = _a.merge(_b, on=['trial_month','size_code'], how='outer').sort_values('trial_month')
    _idx_df['trial_index'] = (_idx_df['ariel_trial'] / _idx_df['attack_trial'].replace(0, np.nan) * 100).round(1)

    n_s=len(comb_sizes); n_c=min(2,n_s); n_r=(n_s+n_c-1)//n_c
    specs = [[{'secondary_y':True}]*n_c for _ in range(n_r)]
    fig_comb = make_subplots(rows=n_r, cols=n_c, specs=specs, subplot_titles=comb_sizes,
                             vertical_spacing=0.15, horizontal_spacing=0.14)

    for idx, size in enumerate(comb_sizes):
        r, c = idx//n_c+1, idx%n_c+1
        for brand in [BOLD_GB, ARIEL_GB]:
            sub = combined_trial[(combined_trial['sub_brand']==brand) & (combined_trial['size_code']==size)].sort_values('trial_month')
            if len(sub)==0: continue
            fig_comb.add_trace(go.Bar(x=sub['trial_month'], y=sub['subbrand_trial_shoppers'],
                name=BRAND_LABEL[brand], marker_color=BRAND_COLOR[brand], opacity=0.85,
                showlegend=(idx==0), legendgroup=BRAND_LABEL[brand]), row=r, col=c, secondary_y=False)

        _s = _idx_df[_idx_df['size_code']==size]
        if len(_s)>0 and _s['trial_index'].notna().any():
            fig_comb.add_trace(go.Scatter(x=_s['trial_month'], y=_s['trial_index'], name='Index Bold/Attack×100',
                mode='lines+markers', line=dict(color='#2CA02C', dash='dash', width=2),
                marker=dict(size=6, symbol='diamond'), showlegend=(idx==0), legendgroup='Index'),
                row=r, col=c, secondary_y=True)
            fig_comb.add_hline(y=100, line_dash='dot', line_color='gray', line_width=1, row=r, col=c, secondary_y=True)

        fig_comb.update_yaxes(title_text='Trial Shoppers', row=r, col=c, secondary_y=False)
        fig_comb.update_yaxes(title_text='Index / Trial Rate%', row=r, col=c, secondary_y=True, showgrid=False)

    fig_comb.update_layout(height=420*n_r, barmode='group',
        title_text='Monthly Trial: Bold vs Ariel per Size', template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0))
    fig_comb.show()

# ── DATA TABLE: Monthly Trial Shoppers — Bold vs Ariel per Size ────
trial_table = _idx_df[['trial_month','size_code','ariel_trial','attack_trial','ariel_rate','trial_index']].copy()
trial_table.columns = ['Month','Size','Bold Trial','Ariel Trial','Bold Trial Rate%','Trial Index']
trial_table = trial_table.sort_values(['Size','Month'])
print('\n📊 DATA TABLE: Monthly Trial — Bold vs Ariel per Size')
print('=' * 100)
print(trial_table.to_string(index=False))
trial_table.to_csv(OUTPUT_DIR / 'pane_b_trial_trend.csv', index=False)


📊 DATA TABLE: Monthly Trial — Bold vs Ariel per Size
     Month          Size  Bold Trial  Ariel Trial  Bold Trial Rate%  Trial Index
2025-01-01          本体通常    19,635.0     11,007.0              45.2        178.4
2025-02-01          本体通常    18,570.0     25,858.0              45.9         71.8
2025-03-01          本体通常    19,196.0     81,318.0              50.1         23.6
2025-04-01          本体通常    50,541.0     24,938.0              58.5        202.7
2025-05-01          本体通常    79,933.0     13,194.0              59.5        605.8
2025-06-01          本体通常    39,120.0     14,415.0              49.5        271.4
2025-07-01          本体通常    24,177.0     13,946.0              45.4        173.4
2025-01-01         詰替超特大         1.0          2.0              50.0         50.0
2025-02-01         詰替超特大         3.0          5.0              50.0         60.0
2025-03-01         詰替超特大         3.0          2.0              75.0        150.0
2025-04-01         詰替超特大         NaN          1.0      

In [24]:
# ── B-3: Head-to-Head Sub-brand Trial by Size ─────────────────────────
h2h = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].groupby(['sub_brand','size_code']).agg(
    subbrand_trial=('subbrand_trial_shoppers','sum'), avg_trial_rate=('trial_rate','mean')).reset_index()
h2h_sizes = order_and_filter_sizes(h2h['size_code'].unique())
h2h = h2h[h2h['size_code'].isin(h2h_sizes)].copy()

_ha = h2h[h2h['sub_brand']==BOLD_GB][['size_code','subbrand_trial','avg_trial_rate']].rename(
    columns={'subbrand_trial':'ariel_trial','avg_trial_rate':'ariel_rate'})
_hb = h2h[h2h['sub_brand']==ARIEL_GB][['size_code','subbrand_trial','avg_trial_rate']].rename(
    columns={'subbrand_trial':'attack_trial','avg_trial_rate':'attack_rate'})
h2h_idx = _ha.merge(_hb, on='size_code', how='outer')
h2h_idx['index'] = (h2h_idx['ariel_trial'] / h2h_idx['attack_trial'].replace(0,np.nan) * 100).round(1)

print('📊 DATA TABLE: Head-to-Head Trial by Size')
print('=' * 80)
print(h2h_idx.to_string(index=False))

fig_h2h = make_subplots(specs=[[{'secondary_y':True}]])
for brand_code, brand_label, color in [(BOLD_GB, 'Bold Gel Ball', '#2980B9'), (ARIEL_GB, 'Ariel Gel Ball', '#E67E22')]:
    d = h2h[h2h['sub_brand']==brand_code]
    fig_h2h.add_trace(go.Bar(x=d['size_code'], y=d['subbrand_trial'], name=brand_label,
        marker_color=color, opacity=0.8, text=[fmt_km(v) for v in d['subbrand_trial']], textposition='outside'),
        secondary_y=False)
fig_h2h.add_trace(go.Scatter(x=h2h_idx['size_code'], y=h2h_idx['index'], name='Index (Bold/Attack×100)',
    mode='lines+markers+text', line=dict(color='#27AE60', dash='dash', width=2.5),
    marker=dict(size=10, symbol='diamond', color='#27AE60'),
    text=[f'{v:.0f}' for v in h2h_idx['index']], textposition='bottom center'), secondary_y=True)
fig_h2h.add_hline(y=100, line_dash='dot', line_color='#BDC3C7', line_width=1.5, secondary_y=True)
fig_h2h.update_layout(title='Head-to-Head: Sub-brand Trial by Size', barmode='group',
    template='plotly_white', height=550, xaxis=dict(categoryorder='array', categoryarray=h2h_sizes))
fig_h2h.update_yaxes(title_text='Trial Shoppers', secondary_y=False)
fig_h2h.update_yaxes(title_text='Index', secondary_y=True, showgrid=False)
fig_h2h.show()

h2h_idx.to_csv(OUTPUT_DIR / 'pane_b_h2h_trial.csv', index=False)


📊 DATA TABLE: Head-to-Head Trial by Size
    size_code  ariel_trial  ariel_rate  attack_trial  attack_rate  index
         本体通常       251172        50.6     184,676.0         50.8  136.0
        詰替超特大           11        65.0          12.0         73.3   91.7
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ            2        66.7           NaN          NaN    NaN
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ        61643        27.0      69,298.0         25.2   89.0


In [25]:
# ── B-4: Price Gap Heatmaps ───────────────────────────────────────────
bold_w = df_asp_weekly[(df_asp_weekly['sub_brand']==BOLD_GB) & (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))]\
    [['week_end','retailer','size_code','weighted_asp','total_units','buyer_count']].copy()
ariel_w = df_asp_weekly[(df_asp_weekly['sub_brand']==ARIEL_GB) & (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))]\
    [['week_end','retailer','size_code','weighted_asp','total_units']].copy()
paired = bold_w.merge(ariel_w, on=['week_end','retailer','size_code'], suffixes=('_ariel','_attack'), how='inner')
paired['bold_asp_bin'] = (paired['weighted_asp_ariel']//50*50).astype(int)
paired['ariel_asp_bin'] = (paired['weighted_asp_attack']//50*50).astype(int)

sizes_for_hm = order_and_filter_sizes(paired['size_code'].unique())
heatmap_tables = []

for size in sizes_for_hm:
    s = paired[paired['size_code']==size].copy()
    if len(s) < MIN_OBS_FREQ: continue

    # Count observations per bin combination
    obs = s.groupby(['ariel_asp_bin','bold_asp_bin']).size().reset_index(name='obs')

    # Compute unit index based on average units sold per store (retailer)
    # Step 1: Average units per retailer within each bin combination
    store_avg = s.groupby(['ariel_asp_bin','bold_asp_bin','retailer']).agg(
        avg_units_ariel=('total_units_ariel','mean'),
        avg_units_attack=('total_units_attack','mean')
    ).reset_index()
    # Step 2: Average across retailers for the bin combination
    bin_agg = store_avg.groupby(['ariel_asp_bin','bold_asp_bin']).agg(
        avg_units_ariel=('avg_units_ariel','mean'),
        avg_units_attack=('avg_units_attack','mean')
    ).reset_index()
    bin_agg = bin_agg.merge(obs, on=['ariel_asp_bin','bold_asp_bin'], how='left')
    # Suppress low-frequency bins
    bin_agg.loc[bin_agg['obs'] < MIN_OBS_FREQ, ['avg_units_ariel','avg_units_attack']] = np.nan
    bin_agg['unit_index'] = bin_agg['avg_units_ariel'] / bin_agg['avg_units_attack'] * 100

    # Pivot for heatmap — Y axis ascending for intuitive reading
    hm = bin_agg.pivot_table(index='ariel_asp_bin', columns='bold_asp_bin', values='unit_index')
    hm = hm.sort_index(ascending=True)
    hm_obs = bin_agg.pivot_table(index='ariel_asp_bin', columns='bold_asp_bin', values='obs')
    hm_obs = hm_obs.reindex(index=hm.index, columns=hm.columns)

    if hm.isnull().all().all(): continue

    # Build custom text: index value + obs count
    custom_text = []
    for i in range(hm.shape[0]):
        row = []
        for j in range(hm.shape[1]):
            val = hm.iloc[i, j]
            n = hm_obs.iloc[i, j]
            if pd.isna(val) or pd.isna(n):
                row.append('')
            else:
                row.append(f'{val:.0f}<br><sub>n={int(n)}</sub>')
        custom_text.append(row)

    fig = go.Figure(data=go.Heatmap(
        z=hm.values, x=[str(c) for c in hm.columns], y=[str(r) for r in hm.index],
        text=custom_text, texttemplate='%{text}', textfont=dict(size=11),
        colorscale='RdYlGn', zmin=50, zmax=150,
        colorbar=dict(title='Unit Index'),
        hovertemplate='Bold ASP: %{x}<br>Ariel ASP: %{y}<br>Index: %{z:.0f}<extra></extra>'
    ))
    fig.update_layout(
        title=f'【{size}】Price Gap × Bold Unit Index vs Ariel (avg units/store, ≥{MIN_OBS_FREQ} obs)',
        xaxis_title='Bold ASP (50JPY bin)', yaxis_title='Ariel ASP (50JPY bin)',
        width=800, height=580, template='plotly_white')
    fig.show()

    # Collect table data
    bin_agg['size_code'] = size
    heatmap_tables.append(bin_agg[['ariel_asp_bin','bold_asp_bin','unit_index','obs','size_code']])

# ── DATA TABLE: Price Gap Heatmap Values ─────────────────────────────
if heatmap_tables:
    hm_all = pd.concat(heatmap_tables, ignore_index=True).dropna(subset=['unit_index'])
    hm_all['unit_index'] = hm_all['unit_index'].round(1)
    print('\n📊 DATA TABLE: Price Gap × Unit Index — per Size (avg units/store)')
    print('=' * 100)
    for size in sizes_for_hm:
        sz_data = hm_all[hm_all['size_code']==size]
        if len(sz_data) == 0: continue
        print(f'\n▶ {size}')
        print(sz_data[['ariel_asp_bin','bold_asp_bin','unit_index','obs']].to_string(index=False))
    hm_all.to_csv(OUTPUT_DIR / 'pane_b_heatmap_values.csv', index=False)


📊 DATA TABLE: Price Gap × Unit Index — per Size (avg units/store)

▶ 本体通常
 ariel_asp_bin  bold_asp_bin  unit_index  obs
           150           150       214.7  137
           150           200        98.7   27
           150           250        27.7  116
           150           300        17.6   82
           150           350        11.0  212
           200           150       318.5   27
           200           200       191.6   80
           200           250        81.3   46
           200           300        38.7   64
           200           350        29.3   24
           200           400        16.8   62
           250           150       693.6  143
           250           200       277.0   73
           250           250       159.6  346
           250           300       155.2   45
           250           350        52.2   41
           300           150       710.6   56
           300           200       565.1   94
           300           250       217.0  102
     

---
# Pane C — Repeat Shopper
*Cohort funnel, size migration, pre/post renewal, ASP band analysis.*


In [26]:
# ── C-1: Cohort Funnel by Entry Size ──────────────────────────────────
funnel = df_cohort.groupby(['sub_brand', 'trial_size', 'outcome']).agg(
    shoppers=('shopper_key', 'nunique')).reset_index()
funnel_pivot = funnel.pivot_table(index=['sub_brand','trial_size'], columns='outcome',
    values='shoppers', fill_value=0).reset_index()
if 'Repeat' in funnel_pivot.columns and 'Lapse' in funnel_pivot.columns:
    funnel_pivot['total'] = funnel_pivot['Repeat'] + funnel_pivot['Lapse']
    funnel_pivot['repeat_rate_%'] = (funnel_pivot['Repeat'] / funnel_pivot['total'] * 100).round(1)
    funnel_pivot['lapse_rate_%'] = (funnel_pivot['Lapse'] / funnel_pivot['total'] * 100).round(1)

print('📊 DATA TABLE: 6-Month Cohort Funnel')
print('=' * 80)
for brand in [BOLD_GB, ARIEL_GB]:
    print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
    print(funnel_pivot[funnel_pivot['sub_brand']==brand].to_string(index=False))

# Bar chart
if 'repeat_rate_%' in funnel_pivot.columns:
    valid_sizes = order_and_filter_sizes(funnel_pivot['trial_size'].unique())
    plot_df = funnel_pivot[funnel_pivot['trial_size'].isin(valid_sizes)].copy()
    plot_df['trial_size'] = pd.Categorical(plot_df['trial_size'], categories=valid_sizes, ordered=True)
    plot_df = plot_df.sort_values('trial_size')
    fig = px.bar(plot_df, x='trial_size', y='repeat_rate_%', color='sub_brand', barmode='group',
        color_discrete_map={BOLD_GB:'#1E90FF', ARIEL_GB:'#FF6347'}, text='repeat_rate_%',
        title='6-Month Repeat Rate by Trial Entry Size — Higher = Better Retention')
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(template='plotly_white', height=500)
    fig.show()

funnel_pivot.to_csv(OUTPUT_DIR / 'pane_c_cohort_funnel.csv', index=False)


📊 DATA TABLE: 6-Month Cohort Funnel

▶ Bold Gel Ball
     sub_brand    trial_size     Lapse   Repeat     total  repeat_rate_%  lapse_rate_%
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常 165,847.0 85,325.0 251,172.0           34.0          66.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ         詰替超特大       9.0      2.0      11.0           18.2          81.8
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ       1.0      1.0       2.0           50.0          50.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ     詰替超ｼﾞｬﾝﾎﾞ      91.0     23.0     114.0           20.2          79.8
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  45,215.0 20,170.0  65,385.0           30.8          69.2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          詰替通常       2.0      0.0       2.0            0.0         100.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ    詰替ﾃﾗｼﾞｬﾝﾎﾞ  12,275.0  5,156.0  17,431.0           29.6          70.4
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ 133,983.0 76,094.0 210,077.0           36.2          63.8
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  41,060.0 20,583.0  61,643.0           33.4          66.6
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ           ｿﾉﾀ     172.0     29.0     201.0           14.4     

In [27]:
# ── C-2: Size Migration Matrix ────────────────────────────────────────
for brand, brand_label, cscale in [(BOLD_GB, 'ボールドジェルボール', 'Blues'), (ARIEL_GB, 'アリエールジェルボール', 'Oranges')]:
    repeat_shoppers = df_cohort[
        (df_cohort['outcome']=='Repeat') & (df_cohort['sub_brand']==brand) &
        (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) & (~df_cohort['repeat_size'].isin(EXCLUDED_SIZES))
    ]
    if len(repeat_shoppers) == 0: continue
    migration = repeat_shoppers.groupby(['trial_size','repeat_size']).agg(shoppers=('shopper_key','nunique')).reset_index()
    migration_matrix = migration.pivot_table(index='trial_size', columns='repeat_size', values='shoppers', fill_value=0)
    ordered = order_and_filter_sizes(list(migration_matrix.index) + list(migration_matrix.columns))
    migration_matrix = migration_matrix.reindex(
        index=[s for s in ordered if s in migration_matrix.index],
        columns=[s for s in ordered if s in migration_matrix.columns], fill_value=0)
    migration_pct = migration_matrix.div(migration_matrix.sum(axis=1), axis=0) * 100

    print(f'\n📊 DATA TABLE: Size Migration — {brand_label} (Trial → Repeat %)')
    print(migration_pct.round(1).to_string())

    fig = px.imshow(migration_pct.values, x=migration_pct.columns.tolist(), y=migration_pct.index.tolist(),
        text_auto='.1f', color_continuous_scale=cscale,
        labels=dict(x='Repeat Size', y='Trial Size', color='%'),
        title=f'[{brand_label}] Size Migration: Where Do Trial Shoppers Repeat? (%)')
    fig.update_layout(height=500, template='plotly_white')
    fig.show()



📊 DATA TABLE: Size Migration — ボールドジェルボール (Trial → Repeat %)
repeat_size    本体通常  詰替超特大  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ
trial_size                             
本体通常           97.4    0.0          2.6
詰替超特大           NaN    NaN          NaN
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   0.0    0.0        100.0
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ    24.2    0.0         75.8



📊 DATA TABLE: Size Migration — アリエールジェルボール (Trial → Repeat %)
repeat_size  本体通常  詰替超特大  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ
trial_size                           
本体通常         95.7    0.0          4.3
詰替超特大         NaN    NaN          NaN
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  12.8    0.0         87.2


In [28]:
# ── C-3: Pre vs Post Renewal Repeat Rate ──────────────────────────────
df_cohort['renewal_period'] = df_cohort['trial_date'].apply(lambda x: 'Pre-Renewal' if x < renewal_date else 'Post-Renewal')
bold_cohort = df_cohort[df_cohort['sub_brand']==BOLD_GB].copy()

period_funnel = bold_cohort.groupby(['renewal_period','trial_size','outcome']).agg(
    shoppers=('shopper_key','nunique')).reset_index()
period_pivot = period_funnel.pivot_table(index=['renewal_period','trial_size'], columns='outcome',
    values='shoppers', fill_value=0).reset_index()
if 'Repeat' in period_pivot.columns and 'Lapse' in period_pivot.columns:
    period_pivot['total'] = period_pivot['Repeat'] + period_pivot['Lapse']
    period_pivot['repeat_rate_%'] = (period_pivot['Repeat'] / period_pivot['total'] * 100).round(1)

print('📊 DATA TABLE: Pre vs Post Renewal Repeat Rate — Bold Gel Ball')
print('=' * 80)
print(period_pivot.to_string(index=False))

if 'repeat_rate_%' in period_pivot.columns:
    fig = px.bar(period_pivot, x='trial_size', y='repeat_rate_%', color='renewal_period', barmode='group',
        text='repeat_rate_%', color_discrete_map={'Pre-Renewal':'#6495ED', 'Post-Renewal':'#FF6347'},
        title='Did the Renewal Improve Repeat Rates? (Bold Gel Ball)')
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(template='plotly_white', height=500, yaxis_title='Repeat Rate (%)')
    fig.show()

period_pivot.to_csv(OUTPUT_DIR / 'pane_c_pre_post_renewal.csv', index=False)


📊 DATA TABLE: Pre vs Post Renewal Repeat Rate — Bold Gel Ball
renewal_period    trial_size    Lapse   Repeat     total  repeat_rate_%
  Post-Renewal          本体通常 96,380.0 46,850.0 143,230.0           32.7
  Post-Renewal         詰替超特大      4.0      0.0       4.0            0.0
  Post-Renewal     詰替超ｼﾞｬﾝﾎﾞ     16.0      4.0      20.0           20.0
  Post-Renewal  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ 15,950.0  7,645.0  23,595.0           32.4
  Post-Renewal          詰替通常      1.0      0.0       1.0            0.0
  Post-Renewal    詰替ﾃﾗｼﾞｬﾝﾎﾞ 11,095.0  4,595.0  15,690.0           29.3
  Post-Renewal 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ 60,573.0 33,878.0  94,451.0           35.9
  Post-Renewal   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ 17,982.0  8,821.0  26,803.0           32.9
  Post-Renewal           ｿﾉﾀ     67.0      7.0      74.0            9.5
   Pre-Renewal          本体通常 69,467.0 38,475.0 107,942.0           35.6
   Pre-Renewal         詰替超特大      5.0      2.0       7.0           28.6
   Pre-Renewal 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      1.0      1.0       2.0           50.0
  

In [29]:
# ── C-3b: Size Migration Direction — Same / Size Down / Size Up ──────
# Stacked bar comparing Bold Gel Ball vs Ariel Gel Ball
# X = trial entry size, Y = % of repeat shoppers by migration direction

def classify_migration(trial_size, repeat_size, size_order):
    """Return migration direction based on SIZE_ORDER index."""
    if trial_size not in size_order or repeat_size not in size_order:
        return 'Other'
    ti, ri = size_order.index(trial_size), size_order.index(repeat_size)
    if ti == ri:
        return '① Same Size'
    elif ri > ti:
        return '③ Size Up (Larger)'
    else:
        return '② Size Down (Smaller)'

MIGRATE_ORDER  = ['① Same Size', '② Size Down (Smaller)', '③ Size Up (Larger)']
MIGRATE_COLORS = {
    '① Same Size':           '#4ECDC4',
    '② Size Down (Smaller)': '#E74C3C',
    '③ Size Up (Larger)':    '#2ECC71',
}

dfs_mig = []
for brand in [BOLD_GB, ARIEL_GB]:
    repeat_shoppers_mig = df_cohort[
        (df_cohort['outcome']=='Repeat') & (df_cohort['sub_brand']==brand) &
        (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) &
        (~df_cohort['repeat_size'].isin(EXCLUDED_SIZES))
    ].copy()
    if len(repeat_shoppers_mig) == 0:
        continue
    repeat_shoppers_mig['migration_type'] = repeat_shoppers_mig.apply(
        lambda row: classify_migration(row['trial_size'], row['repeat_size'], SIZE_ORDER), axis=1
    )
    tmp_agg = repeat_shoppers_mig.groupby(['trial_size', 'migration_type']).agg(
        shoppers=('shopper_key', 'nunique')
    ).reset_index()
    tmp_agg['brand'] = brand
    dfs_mig.append(tmp_agg)

if dfs_mig:
    mig_compare = pd.concat(dfs_mig, ignore_index=True)
    mig_compare = mig_compare[mig_compare['migration_type'] != 'Other']

    # % within each brand × trial_size bucket
    totals = mig_compare.groupby(['brand', 'trial_size'])['shoppers'].transform('sum')
    mig_compare['pct'] = (mig_compare['shoppers'] / totals * 100).round(1)

    # Apply SIZE_ORDER ordering
    valid_sizes = order_and_filter_sizes(mig_compare['trial_size'].unique())
    mig_compare['trial_size'] = pd.Categorical(mig_compare['trial_size'], categories=valid_sizes, ordered=True)
    mig_compare = mig_compare[mig_compare['trial_size'].notna()].sort_values('trial_size')

    brands_list = [BOLD_GB, ARIEL_GB]
    brands_available = [b for b in brands_list if b in mig_compare['brand'].values]

    fig_mig = make_subplots(
        rows=len(brands_available), cols=1,
        subplot_titles=[BRAND_LABEL.get(b, b) for b in brands_available],
        vertical_spacing=0.18
    )

    for row_idx, brand in enumerate(brands_available, 1):
        bd = mig_compare[mig_compare['brand'] == brand]
        for mtype in MIGRATE_ORDER:
            sub = bd[bd['migration_type'] == mtype]
            if len(sub) == 0:
                continue
            fig_mig.add_trace(go.Bar(
                x=sub['trial_size'].astype(str),
                y=sub['pct'],
                name=mtype,
                marker_color=MIGRATE_COLORS[mtype],
                text=sub['pct'].apply(lambda v: f'{v:.1f}%'),
                textposition='inside',
                insidetextanchor='middle',
                legendgroup=mtype,
                showlegend=(row_idx == 1)
            ), row=row_idx, col=1)

    fig_mig.update_layout(
        height=420 * len(brands_available),
        title_text=(
            'Size Migration Direction: Bold Gel Ball vs Ariel Gel Ball<br>'
            '<sup>% of repeat shoppers — Same size vs Size Down (smaller) vs Size Up (larger refill)</sup>'
        ),
        barmode='stack',
        template='plotly_white',
        legend=dict(orientation='h', y=-0.05)
    )
    fig_mig.update_yaxes(title_text='% of Repeat Shoppers', range=[0, 105])
    fig_mig.update_xaxes(title_text='Trial Entry Size')
    fig_mig.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    print('\n📊 DATA TABLE: Size Migration Direction Summary')
    print('=' * 100)
    summary_tbl = mig_compare.pivot_table(
        index=['brand', 'trial_size'],
        columns='migration_type',
        values=['pct', 'shoppers'],
        fill_value=0
    ).round(1)
    print(summary_tbl.to_string())
    mig_compare.to_csv(OUTPUT_DIR / 'pane_c_migration_direction.csv', index=False)
else:
    print('⚠️ No migration data available')


📊 DATA TABLE: Size Migration Direction Summary
                                     pct                                             shoppers                                         
migration_type               ① Same Size ② Size Down (Smaller) ③ Size Up (Larger) ① Same Size ② Size Down (Smaller) ③ Size Up (Larger)
brand          trial_size                                                                                                             
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  本体通常                 95.7                   0.0                4.3    34,689.0                   0.0            1,573.0
               詰替ﾒｶﾞｼﾞｬﾝﾎﾞ          87.2                  12.8                0.0     8,292.0               1,216.0                0.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 本体通常                 97.4                   0.0                2.6    64,103.0                   0.0            1,687.0
               詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         0.0                   0.0              100.0         0.0                   0.0                1.0
       

In [30]:
# ── C-3c: Trial ASP Distribution — Repeat vs Lapse (Box Plots) ───────
# Shows whether repeaters and lapsers entered at different price points

bold_boxplot_data = df_cohort[
    (df_cohort['sub_brand']==BOLD_GB) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) &
    (df_cohort['trial_asp'].notna())
].copy()

box_sizes = order_and_filter_sizes(bold_boxplot_data['trial_size'].unique())
if box_sizes:
    n_s = len(box_sizes); n_c = min(2, n_s); n_r = (n_s + n_c - 1) // n_c
    fig_box = make_subplots(rows=n_r, cols=n_c, subplot_titles=[f'Bold {s}' for s in box_sizes],
                            vertical_spacing=0.16, horizontal_spacing=0.14)

    BOX_COLORS = {'Repeat': '#2ECC71', 'Lapse': '#E74C3C'}
    for idx, size in enumerate(box_sizes):
        r, c = idx // n_c + 1, idx % n_c + 1
        for outcome in ['Repeat', 'Lapse']:
            subset = bold_boxplot_data[
                (bold_boxplot_data['trial_size']==size) &
                (bold_boxplot_data['outcome']==outcome)
            ]
            if len(subset) == 0: continue
            fig_box.add_trace(go.Box(
                y=subset['trial_asp'], name=outcome,
                marker_color=BOX_COLORS[outcome],
                boxmean='sd',
                showlegend=(idx == 0),
                legendgroup=outcome,
            ), row=r, col=c)
        fig_box.update_yaxes(title_text='Trial ASP (JPY)', row=r, col=c)

    fig_box.update_layout(
        height=420 * n_r,
        title_text=(
            'Trial ASP Distribution: Repeat vs Lapse Shoppers (Bold Gel Ball)<br>'
            '<sup>Did repeat shoppers enter at different prices than lapsers?</sup>'
        ),
        template='plotly_white',
        legend=dict(orientation='h', y=-0.04)
    )
    fig_box.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    box_stats = bold_boxplot_data.groupby(['trial_size', 'outcome'])['trial_asp'].agg(
        ['count', 'mean', 'median', 'std', 'min', 'max']
    ).round(0).reset_index()
    box_stats.columns = ['Size', 'Outcome', 'Shoppers', 'Mean ASP', 'Median ASP', 'Std', 'Min', 'Max']
    print('\n📊 DATA TABLE: Trial ASP Statistics — Repeat vs Lapse')
    print('=' * 100)
    print(box_stats.to_string(index=False))
    box_stats.to_csv(OUTPUT_DIR / 'pane_c_asp_boxplot_stats.csv', index=False)
else:
    print('⚠️ No data for box plots')


📊 DATA TABLE: Trial ASP Statistics — Repeat vs Lapse
         Size Outcome  Shoppers  Mean ASP  Median ASP   Std     Min     Max
         本体通常   Lapse    165847     245.0       199.0  84.0     0.0   679.0
         本体通常  Repeat     85325     240.0       199.0  78.0     0.0   519.0
        詰替超特大   Lapse         9     406.0       349.0 109.0   332.0   662.0
        詰替超特大  Repeat         2     560.0       560.0  53.0   523.0   598.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   Lapse         1   1,408.0     1,408.0   NaN 1,408.0 1,408.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  Repeat         1   1,408.0     1,408.0   NaN 1,408.0 1,408.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse     45215   2,444.0     2,531.0 473.0     0.0 3,278.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat     20170   2,432.0     2,480.0 422.0     0.0 3,278.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ   Lapse     12275   3,088.0     3,077.0 401.0     0.0 3,608.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  Repeat      5156   2,997.0     2,998.0 391.0     0.0 3,608.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   Lapse    133983     890.0       931.0 180.0     0.0 1,280.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  Repeat     76094   

In [31]:
# ── C-4: ASP Band — Trial/Repeat/Lapse Rates ─────────────────────────
bold_trials = df_cohort[(df_cohort['sub_brand']==BOLD_GB) & (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))].copy()
bold_trials['asp_band'] = (bold_trials['trial_asp'] // 50 * 50).astype('Int64')

asp_bin_agg = bold_trials.groupby(['trial_size','asp_band']).agg(
    trial_shoppers=('shopper_key','nunique'), obs_count=('shopper_key','count')).reset_index()
asp_bin_split = bold_trials.groupby(['trial_size','asp_band','outcome']).agg(
    shoppers=('shopper_key','nunique')).reset_index()

repeat_counts = asp_bin_split[asp_bin_split['outcome']=='Repeat'][['trial_size','asp_band','shoppers']].rename(columns={'shoppers':'repeat_shoppers'})
lapse_counts = asp_bin_split[asp_bin_split['outcome']=='Lapse'][['trial_size','asp_band','shoppers']].rename(columns={'shoppers':'lapse_shoppers'})

rate_df = asp_bin_agg.merge(df_universe, on=['trial_size','asp_band'], how='left')\
    .merge(repeat_counts, on=['trial_size','asp_band'], how='left')\
    .merge(lapse_counts, on=['trial_size','asp_band'], how='left')
rate_df[['repeat_shoppers','lapse_shoppers']] = rate_df[['repeat_shoppers','lapse_shoppers']].fillna(0)
rate_df['trial_rate_%'] = (rate_df['trial_shoppers'] / rate_df['all_shoppers'] * 100).round(1)
rate_df['repeat_rate_%'] = (rate_df['repeat_shoppers'] / rate_df['trial_shoppers'] * 100).round(1)
rate_df['lapse_rate_%'] = (rate_df['lapse_shoppers'] / rate_df['trial_shoppers'] * 100).round(1)
for col in ['trial_rate_%','repeat_rate_%','lapse_rate_%']:
    rate_df[col] = rate_df[col].clip(upper=100)

print('📊 DATA TABLE: Trial/Repeat/Lapse Rate by ASP Band (sample)')
print(rate_df[['trial_size','asp_band','all_shoppers','trial_shoppers','trial_rate_%','repeat_rate_%','lapse_rate_%']].head(15).to_string(index=False))

sizes_rate = order_and_filter_sizes(rate_df['trial_size'].unique())
if sizes_rate:
    n_s=len(sizes_rate); n_c=min(2,n_s); n_r=(n_s+n_c-1)//n_c
    specs = [[{"secondary_y":True}]*n_c for _ in range(n_r)]
    fig_rate = make_subplots(rows=n_r, cols=n_c, specs=specs,
        subplot_titles=[f'Bold {s}' for s in sizes_rate], vertical_spacing=0.16, horizontal_spacing=0.14)
    RATE_COLORS = {'trial_rate_%':'#1E90FF', 'repeat_rate_%':'#2ECC71', 'lapse_rate_%':'#E74C3C'}
    for idx, size in enumerate(sizes_rate):
        r, c = idx//n_c+1, idx%n_c+1
        subset = rate_df[rate_df['trial_size']==size].sort_values('asp_band')
        if len(subset)==0: continue
        x_labels = subset['asp_band'].astype(str) + '~'
        fig_rate.add_trace(go.Bar(x=x_labels, y=subset['all_shoppers'], name='All Shoppers',
            marker_color='#D0D0D0', opacity=0.6, showlegend=(idx==0), legendgroup='all_vol'), row=r, col=c, secondary_y=False)
        for rate_col, color in RATE_COLORS.items():
            fig_rate.add_trace(go.Scatter(x=x_labels, y=subset[rate_col], name=rate_col.replace('_',' '),
                mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=6),
                showlegend=(idx==0), legendgroup=rate_col), row=r, col=c, secondary_y=True)
    fig_rate.update_layout(height=440*n_r, title_text='ASP Band — Trial/Repeat/Lapse Rates (Bold by Size)',
        template='plotly_white', legend=dict(orientation='h', y=-0.04, x=0.5, xanchor='center'))
    fig_rate.update_xaxes(title_text='ASP Band (JPY)')
    for _r in range(1, n_r+1):
        for _c in range(1, n_c+1):
            fig_rate.update_yaxes(title_text='All Shoppers', secondary_y=False, row=_r, col=_c)
            fig_rate.update_yaxes(title_text='Rate (%)', secondary_y=True, row=_r, col=_c, range=[0,105], showgrid=False)
    fig_rate.show()

rate_df.to_csv(OUTPUT_DIR / 'pane_c_asp_band_rates.csv', index=False)


📊 DATA TABLE: Trial/Repeat/Lapse Rate by ASP Band (sample)
trial_size  asp_band  all_shoppers  trial_shoppers  trial_rate_%  repeat_rate_%  lapse_rate_%
      本体通常         0          9023            2536          28.1           31.5          68.5
      本体通常        50          7292            4749          65.1           26.1          73.9
      本体通常       100          8201            3244          39.6           32.3          67.7
      本体通常       150        250766          127104          50.7           36.7          63.3
      本体通常       200        139249           23015          16.5           26.9          73.1
      本体通常       250        116001           43113          37.2           35.2          64.8
      本体通常       300         54817           11198          20.4           30.8          69.2
      本体通常       350        116837           29045          24.9           31.1          68.9
      本体通常       400         26199            7162          27.3           25.0          75.0
 

In [32]:
# ── C-5: Trial/Repeat/Lapse Price-Point Productivity (per store) + Frequency ──
# 3 charts: ALL trial, REPEAT only, LAPSE only — each faceted by size
# Blue bars = avg shoppers per store (left Y), Orange line = store execution freq (right Y)
# Per-store normalisation: shoppers / total_stores at that ASP band
# store_exec_freq = SUM(n_stores) across weeks = week × #stores executing that price point

bold_pp = df_cohort[
    (df_cohort['sub_brand']==BOLD_GB) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))
].copy()
bold_pp['asp_band'] = (bold_pp['trial_asp'] // 50 * 50).astype('Int64')

# ── Build store execution frequency from df_asp_weekly (week × # stores per ASP band × size) ──
bold_weekly = df_asp_weekly[
    (df_asp_weekly['sub_brand']==BOLD_GB) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
].copy()
bold_weekly['asp_band'] = (bold_weekly['weighted_asp'] // 50 * 50).astype('Int64')
market_freq = bold_weekly.groupby(['size_code', 'asp_band']).agg(
    store_exec_freq=('n_stores', 'sum')   # SUM of stores executing that price point across weeks
).reset_index().rename(columns={'size_code': 'trial_size'})

# Aggregate: all trial + split by outcome
pp_all = bold_pp.groupby(['trial_size', 'asp_band']).agg(
    trial_shoppers=('shopper_key', 'nunique')
).reset_index()

pp_split = bold_pp.groupby(['trial_size', 'asp_band', 'outcome']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index()

# Merge store execution frequency onto shopper tables
pp_all = pp_all.merge(market_freq, on=['trial_size', 'asp_band'], how='left')
pp_split = pp_split.merge(market_freq, on=['trial_size', 'asp_band'], how='left')

# Compute avg shoppers per store — 2 significant figures (有効数字第二位)
def _to_2sf(x):
    if pd.isna(x) or x == 0:
        return x
    return float(f'{x:.2g}')

pp_all['shoppers_per_store'] = (pp_all['trial_shoppers'] / pp_all['store_exec_freq']).apply(_to_2sf)
pp_split['shoppers_per_store'] = (pp_split['shoppers'] / pp_split['store_exec_freq']).apply(_to_2sf)

# Filter out ASP bands with store execution freq < 100
MIN_STORE_EXEC = 10000
pp_all = pp_all[pp_all['store_exec_freq'] >= MIN_STORE_EXEC].reset_index(drop=True)
pp_split = pp_split[pp_split['store_exec_freq'] >= MIN_STORE_EXEC].reset_index(drop=True)

# Additional filter: 25th-pct threshold (min=5 shoppers) to skip low-data bands
pp_all_filt = (
    pp_all.groupby('trial_size', group_keys=False)
    .apply(lambda g: g[g['trial_shoppers'] >= max(g['trial_shoppers'].quantile(0.25), 5)])
    .reset_index(drop=True)
)
pp_split_filt = (
    pp_split.groupby(['trial_size', 'outcome'], group_keys=False)
    .apply(lambda g: g[g['shoppers'] >= max(g['shoppers'].quantile(0.25), 5)])
    .reset_index(drop=True)
)

sizes_pp = order_and_filter_sizes(pp_all_filt['trial_size'].unique())

def make_asp_bin_chart(data_df, y_col, freq_col, sizes, title, y_label):
    """Bar chart of avg shoppers/store by ASP band; store execution freq on secondary Y."""
    if len(sizes) == 0:
        print(f'⚠️ No data for: {title}')
        return
    n_s = len(sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c
    specs = [[{"secondary_y": True} for _ in range(n_c)] for _ in range(n_r)]
    fig = make_subplots(
        rows=n_r, cols=n_c, specs=specs,
        subplot_titles=[f'Bold {s}' for s in sizes],
        vertical_spacing=0.16, horizontal_spacing=0.14
    )
    for idx, size in enumerate(sizes):
        r = idx // n_c + 1
        c = idx % n_c + 1
        subset = data_df[data_df['trial_size'] == size].sort_values('asp_band')
        if len(subset) == 0: continue
        x_labels = subset['asp_band'].astype(str) + '~'
        fig.add_trace(go.Bar(
            x=x_labels, y=subset[y_col], name=y_label,
            marker_color='#4C9BE8', opacity=0.8,
            text=[f'{v:.2g}' for v in subset[y_col]],
            textposition='outside', textfont=dict(size=10),
            showlegend=(idx == 0), legendgroup='shoppers'
        ), row=r, col=c, secondary_y=False)
        fig.add_trace(go.Scatter(
            x=x_labels, y=subset[freq_col], name='Store Execution Freq (week × #stores)',
            mode='lines+markers', line=dict(color='#FF6347', width=2),
            marker=dict(size=6, symbol='circle'),
            showlegend=(idx == 0), legendgroup='frequency'
        ), row=r, col=c, secondary_y=True)
    fig.update_layout(
        height=420 * n_r, title_text=title, template='plotly_white',
        legend=dict(orientation='h', y=-0.04, x=0.5, xanchor='center')
    )
    fig.update_xaxes(title_text='ASP (50 JPY bin)')
    for row in range(1, n_r + 1):
        for col in range(1, n_c + 1):
            fig.update_yaxes(title_text=y_label, secondary_y=False, row=row, col=col)
            fig.update_yaxes(title_text='Store Exec Freq', secondary_y=True, row=row, col=col, showgrid=False)
    fig.show()

# ── Chart 1: All trial shoppers (per store) ──────────────────────────
make_asp_bin_chart(
    data_df=pp_all_filt, y_col='shoppers_per_store', freq_col='store_exec_freq', sizes=sizes_pp,
    title=('ボールドジェルボール: Trial Entry ASP Band (50JPY) — Avg Trial Shoppers per Store<br>'
           '<sup>Blue bars = avg trial shoppers / store | Orange line = store execution freq (week × #stores)</sup>'),
    y_label='Avg Trial Shoppers / Store'
)

# ── Chart 2: Repeat shoppers only (per store) ────────────────────────
repeat_pp = pp_split_filt[pp_split_filt['outcome'] == 'Repeat']
sizes_repeat_pp = order_and_filter_sizes(repeat_pp['trial_size'].unique())
make_asp_bin_chart(
    data_df=repeat_pp, y_col='shoppers_per_store', freq_col='store_exec_freq', sizes=sizes_repeat_pp,
    title=('ボールドジェルボール: Trial ASP Band — Avg Repeat Shoppers per Store (6-month repeat)<br>'
           '<sup>Blue bars = avg repeat shoppers / store | Orange line = store execution freq (week × #stores)</sup>'),
    y_label='Avg Repeat Shoppers / Store'
)

# ── Chart 3: Lapse shoppers only (per store) ─────────────────────────
lapse_pp = pp_split_filt[pp_split_filt['outcome'] == 'Lapse']
sizes_lapse_pp = order_and_filter_sizes(lapse_pp['trial_size'].unique())
make_asp_bin_chart(
    data_df=lapse_pp, y_col='shoppers_per_store', freq_col='store_exec_freq', sizes=sizes_lapse_pp,
    title=('ボールドジェルボール: Trial ASP Band — Avg Lapse Shoppers per Store (no repeat within 6 months)<br>'
           '<sup>Blue bars = avg lapse shoppers / store | Orange line = store execution freq (week × #stores)</sup>'),
    y_label='Avg Lapse Shoppers / Store'
)

# ── DATA TABLES ──────────────────────────────────────────────────────
print('\n📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — ALL Trial')
print('=' * 110)
print(pp_all_filt[['trial_size','asp_band','trial_shoppers','store_exec_freq','shoppers_per_store']].to_string(index=False))

print('\n📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — REPEAT only')
print('=' * 100)
print(repeat_pp[['trial_size','asp_band','shoppers','store_exec_freq','shoppers_per_store']].to_string(index=False))

print('\n📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — LAPSE only')
print('=' * 100)
print(lapse_pp[['trial_size','asp_band','shoppers','store_exec_freq','shoppers_per_store']].to_string(index=False))

pp_all_filt.to_csv(OUTPUT_DIR / 'pane_c_pp_all.csv', index=False)
pp_split_filt.to_csv(OUTPUT_DIR / 'pane_c_pp_split.csv', index=False)


📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — ALL Trial
   trial_size  asp_band  trial_shoppers  store_exec_freq  shoppers_per_store
         本体通常       150          127104        125,489.0                 1.0
         本体通常       200           23015        152,763.0                 0.1
         本体通常       250           43113        157,771.0                 0.3
         本体通常       350           29045        101,270.0                 0.3
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2100            4443         12,261.0                 0.4
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2150            1274         43,198.0                 0.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2250            9435         40,834.0                 0.2
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2300            1548         43,880.0                 0.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2350            1992         39,812.0                 0.1
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2450            2159         11,491.0                 0.2
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2500            9116         16,595.0                 0.6
 

In [33]:
# ── C-5b: Total Shoppers (Repeat+Lapse) per Store + Repeat Rate by ASP Band ──
# Blue bars = total shoppers per store (left Y), Orange line = repeat rate % (right Y)

# Build total (repeat + lapse) per ASP band × size
pp_total = pp_split_filt.groupby(['trial_size', 'asp_band']).agg(
    total_shoppers=('shoppers', 'sum'),
    store_exec_freq=('store_exec_freq', 'first')
).reset_index()

# Get repeat count per ASP band × size
pp_repeat_only = pp_split_filt[pp_split_filt['outcome'] == 'Repeat'][['trial_size', 'asp_band', 'shoppers']].rename(
    columns={'shoppers': 'repeat_shoppers'}
)

pp_total = pp_total.merge(pp_repeat_only, on=['trial_size', 'asp_band'], how='left')
pp_total['repeat_shoppers'] = pp_total['repeat_shoppers'].fillna(0)

# Compute metrics
pp_total['total_per_store'] = (pp_total['total_shoppers'] / pp_total['store_exec_freq']).apply(_to_2sf)
pp_total['repeat_rate'] = (pp_total['repeat_shoppers'] / pp_total['total_shoppers'] * 100).round(1)

sizes_total = order_and_filter_sizes(pp_total['trial_size'].unique())

def make_total_repeat_rate_chart(data_df, sizes, title):
    """Bar chart of total shoppers/store by ASP band; repeat rate % on secondary Y."""
    if len(sizes) == 0:
        print(f'⚠️ No data for: {title}')
        return
    n_s = len(sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c
    specs = [[{"secondary_y": True} for _ in range(n_c)] for _ in range(n_r)]
    fig = make_subplots(
        rows=n_r, cols=n_c, specs=specs,
        subplot_titles=[f'Bold {s}' for s in sizes],
        vertical_spacing=0.16, horizontal_spacing=0.14
    )
    for idx, size in enumerate(sizes):
        r = idx // n_c + 1
        c = idx % n_c + 1
        subset = data_df[data_df['trial_size'] == size].sort_values('asp_band')
        if len(subset) == 0: continue
        x_labels = subset['asp_band'].astype(str) + '~'
        fig.add_trace(go.Bar(
            x=x_labels, y=subset['total_per_store'], name='Total Shoppers / Store',
            marker_color='#4C9BE8', opacity=0.8,
            text=[f'{v:.2g}' for v in subset['total_per_store']],
            textposition='outside', textfont=dict(size=10),
            showlegend=(idx == 0), legendgroup='total'
        ), row=r, col=c, secondary_y=False)
        fig.add_trace(go.Scatter(
            x=x_labels, y=subset['repeat_rate'], name='Repeat Rate (%)',
            mode='lines+markers', line=dict(color='#FF6347', width=2),
            marker=dict(size=6, symbol='circle'),
            showlegend=(idx == 0), legendgroup='rate'
        ), row=r, col=c, secondary_y=True)
    fig.update_layout(
        height=420 * n_r, title_text=title, template='plotly_white',
        legend=dict(orientation='h', y=-0.04, x=0.5, xanchor='center')
    )
    fig.update_xaxes(title_text='ASP (50 JPY bin)')
    for row in range(1, n_r + 1):
        for col in range(1, n_c + 1):
            fig.update_yaxes(title_text='Total Shoppers / Store', secondary_y=False, row=row, col=col)
            fig.update_yaxes(title_text='Repeat Rate (%)', secondary_y=True, row=row, col=col, showgrid=False)
    fig.show()

make_total_repeat_rate_chart(
    data_df=pp_total, sizes=sizes_total,
    title=('ボールドジェルボール: Trial ASP Band — Total Shoppers (Repeat+Lapse) per Store + Repeat Rate<br>'
           '<sup>Blue bars = avg total shoppers / store | Orange line = repeat rate (repeat / total × 100)</sup>')
)

# ── DATA TABLE ──────────────────────────────────────────────────────
print('\n📊 DATA TABLE: ASP Band × Total Shoppers/Store + Repeat Rate')
print('=' * 110)
print(pp_total[['trial_size','asp_band','total_shoppers','repeat_shoppers','store_exec_freq','total_per_store','repeat_rate']].to_string(index=False))


📊 DATA TABLE: ASP Band × Total Shoppers/Store + Repeat Rate
   trial_size  asp_band  total_shoppers  repeat_shoppers  store_exec_freq  total_per_store  repeat_rate
         本体通常       150          127104         46,594.0        125,489.0              1.0         36.7
         本体通常       200           23015          6,202.0        152,763.0              0.1         26.9
         本体通常       250           43113         15,157.0        157,771.0              0.3         35.2
         本体通常       350           29045          9,045.0        101,270.0              0.3         31.1
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2100            4443          1,631.0         12,261.0              0.4         36.7
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2150             979              0.0         43,198.0              0.0          0.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2200             334            334.0         43,071.0              0.0        100.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2250            9435          3,528.0         40,834.0              0.2         37.4
 詰替

---
# Pane D — Lapsed Shopper
*Lapse rate by size and ASP band, post-lapse destination tracking.*


In [34]:
# ── D-1: Lapse Rate by Size ──────────────────────────────────────────
df_active_filt = df_active[~df_active['size_code'].isin(EXCLUDED_SIZES)]

for brand, brand_label in [(BOLD_GB, 'Bold Gel Ball'), (ARIEL_GB, 'Ariel Gel Ball')]:
    lapse_by_size = df_lapsed[(df_lapsed['sub_brand']==brand) & (~df_lapsed['last_size'].isin(EXCLUDED_SIZES))].groupby('last_size').agg(
        lapsed_shoppers=('shopper_key','nunique')).reset_index().rename(columns={'last_size':'size_code'})
    active_brand = df_active_filt[df_active_filt['sub_brand']==brand]
    lapse_rate = lapse_by_size.merge(active_brand[['size_code','active_shoppers']], on='size_code', how='left')
    lapse_rate['lapse_rate_%'] = (lapse_rate['lapsed_shoppers'] / lapse_rate['active_shoppers'] * 100).round(1)
    lapse_rate['_sort'] = lapse_rate['size_code'].map({s:i for i,s in enumerate(SIZE_ORDER)}).fillna(99)
    lapse_rate = lapse_rate.sort_values('_sort').drop(columns='_sort')

    print(f'\n📊 DATA TABLE: Lapse Rate by Size — {brand_label}')
    print('=' * 60)
    print(lapse_rate.to_string(index=False))



📊 DATA TABLE: Lapse Rate by Size — Bold Gel Ball
    size_code  lapsed_shoppers  active_shoppers  lapse_rate_%
         本体通常           226751           364160          62.3
        詰替超特大               13               22          59.1
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                1                4          25.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            74170           173043          42.9
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ            80552           176958          45.5
   詰替ﾃﾗｼﾞｬﾝﾎﾞ            22643            50348          45.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ           231456           465754          49.7

📊 DATA TABLE: Lapse Rate by Size — Ariel Gel Ball
    size_code  lapsed_shoppers  active_shoppers  lapse_rate_%
         本体通常           177822           268713          66.2
        詰替超特大               12               18          66.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            86811           201133          43.2
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ            81217           182328          44.5
   詰替ﾃﾗｼﾞｬﾝﾎﾞ            32908            70448          46.7
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ           213632 

In [35]:
# # ── D-2: Lapse Rate by ASP Band (Ariel, dual-filtered) ───────────────
# df_at_risk_filt = df_at_risk[~df_at_risk['size_code'].isin(EXCLUDED_SIZES)].copy()
# df_at_risk_filt['asp_band'] = (df_at_risk_filt['asp'] // 50 * 50).astype(int)

# asp_lapse = df_at_risk_filt.groupby(['size_code','asp_band']).agg(
#     total_shoppers=('shopper_key','nunique'), lapsed_shoppers=('is_lapsed','sum')).reset_index()
# asp_lapse['lapse_rate_%'] = (asp_lapse['lapsed_shoppers'] / asp_lapse['total_shoppers'] * 100).round(1)
# asp_lapse = asp_lapse.merge(df_week_store[['size_code','asp_band','week_store_count']], on=['size_code','asp_band'], how='left')
# asp_lapse['week_store_count'] = asp_lapse['week_store_count'].fillna(0).astype(int)
# asp_lapse_plot = asp_lapse[(asp_lapse['total_shoppers']>=MIN_FREQ) & (asp_lapse['week_store_count']>=MIN_WEEK_STORE)]

# # All sizes combined
# asp_lapse_all = df_at_risk_filt.groupby('asp_band').agg(
#     total_shoppers=('shopper_key','nunique'), lapsed_shoppers=('is_lapsed','sum')).reset_index()
# asp_lapse_all['lapse_rate_%'] = (asp_lapse_all['lapsed_shoppers'] / asp_lapse_all['total_shoppers'] * 100).round(1)
# asp_lapse_all['size_code'] = '(All Sizes)'
# df_ws_all = df_week_store.groupby('asp_band')['week_store_count'].sum().reset_index()
# asp_lapse_all = asp_lapse_all.merge(df_ws_all, on='asp_band', how='left')
# asp_lapse_all['week_store_count'] = asp_lapse_all['week_store_count'].fillna(0).astype(int)
# asp_lapse_all_plot = asp_lapse_all[(asp_lapse_all['total_shoppers']>=MIN_FREQ) & (asp_lapse_all['week_store_count']>=MIN_WEEK_STORE)]

# print('📊 DATA TABLE: Lapse Rate by ASP Band — All Sizes')
# print(asp_lapse_all_plot.to_string(index=False))

# # Chart
# combined_plot = pd.concat([asp_lapse_all_plot, asp_lapse_plot], ignore_index=True)
# plot_sizes = ['(All Sizes)'] + order_and_filter_sizes(asp_lapse_plot['size_code'].unique())
# n_s=len(plot_sizes); n_c=min(2,n_s); n_r=(n_s+n_c-1)//n_c
# fig = make_subplots(rows=n_r, cols=n_c, subplot_titles=[f'{s}' for s in plot_sizes], vertical_spacing=0.10)
# for idx, size in enumerate(plot_sizes):
#     r, c = idx//n_c+1, idx%n_c+1
#     subset = combined_plot[combined_plot['size_code']==size].sort_values('asp_band')
#     if len(subset)==0: continue
#     fig.add_trace(go.Bar(x=subset['asp_band'].astype(str)+'~', y=subset['lapse_rate_%'],
#         marker_color='#FF6B6B',
#         text=[f'{r:.0f}%\n({int(l)}/{int(t)})' for r,l,t in zip(subset['lapse_rate_%'],subset['lapsed_shoppers'],subset['total_shoppers'])],
#         textposition='outside', showlegend=False), row=r, col=c)
#     fig.update_xaxes(title_text='ASP (50 JPY bin)', row=r, col=c)
#     fig.update_yaxes(title_text='Lapse Rate (%)', row=r, col=c)
# fig.update_layout(height=350*n_r, title_text=f'Lapse Rate by ASP Band (shoppers≥{MIN_FREQ}, wk×store≥{MIN_WEEK_STORE})',
#     template='plotly_white')
# fig.show()

# asp_lapse.to_csv(OUTPUT_DIR / 'pane_d_lapse_by_asp.csv', index=False)


In [36]:
# ── D-3: Post-Lapse Destination (Ariel) ───────────────────────────────
bold_dest = df_destination[df_destination['source_brand']==BOLD_GB]
bold_lapsed_full = df_lapsed[df_lapsed['sub_brand']==BOLD_GB]

dest_summary = bold_dest.groupby(['next_sub_brand','next_size']).agg(shoppers=('shopper_key','nunique')).reset_index().sort_values('shoppers', ascending=False)
total_lapsed = bold_lapsed_full['shopper_key'].nunique()
total_tracked = dest_summary['shoppers'].sum()
category_exit = total_lapsed - total_tracked

print('📊 DATA TABLE: Post-Lapse Destination (Ariel)')
print(f'Total lapsed: {total_lapsed:,} | Tracked: {total_tracked:,} | Category exit: {category_exit:,}')
print('=' * 80)
dest_summary['share_%'] = (dest_summary['shoppers'] / total_lapsed * 100).round(1)
print(dest_summary.head(15).to_string(index=False))

# Treemap
tree_data = pd.concat([dest_summary, pd.DataFrame([{
    'next_sub_brand':'(Category Exit)', 'next_size':'No Purchase',
    'shoppers':category_exit, 'share_%':round(category_exit/total_lapsed*100,1)}])], ignore_index=True)
fig = px.treemap(tree_data, path=['next_sub_brand','next_size'], values='shoppers',
    title='Post-Lapse Destination: Where Do Lost Bold Shoppers Go?',
    color='shoppers', color_continuous_scale='Reds')
fig.update_layout(height=600)
fig.show()

dest_summary.to_csv(OUTPUT_DIR / 'pane_d_destination.csv', index=False)


📊 DATA TABLE: Post-Lapse Destination (Ariel)
Total lapsed: 626,767 | Tracked: 325,821 | Category exit: 300,946
next_sub_brand     next_size  shoppers  share_%
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     21784      3.5
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     21225      3.4
      ｱﾀｯｸ抗菌EX         詰替超特大     15578      2.5
     ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大     14000      2.2
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          本体通常     13709      2.2
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     13082      2.1
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         詰替超特大     12909      2.1
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         詰替超特大     11456      1.8
     ｱﾘｴｰﾙｼﾞｪﾙ          本体通常     10366      1.7
     ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      8870      1.4
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      8468      1.4
      ｱﾀｯｸ抗菌EX          本体通常      8462      1.4
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          本体通常      8221      1.3
          ｴﾏｰﾙ         詰替超特大      8037      1.3
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      6877      1.1


In [37]:
# ── D-3b: Per-Size Destination Treemaps (Ariel) ──────────────────────
# Individual treemap for each Bold exit size showing where lapsed shoppers go

ariel_lapsed_with_size = df_lapsed[
    (df_lapsed['sub_brand']==BOLD_GB) & (~df_lapsed['last_size'].isin(EXCLUDED_SIZES))
]
bold_dest_with_size = df_destination[df_destination['source_brand']==BOLD_GB]

lapse_sizes = order_and_filter_sizes(ariel_lapsed_with_size['last_size'].unique())
per_size_dest_rows = []

for lapse_size in lapse_sizes:
    # Get lapsed shoppers from this size
    lapsed_keys = ariel_lapsed_with_size[ariel_lapsed_with_size['last_size']==lapse_size]['shopper_key']
    size_dest = bold_dest_with_size[bold_dest_with_size['shopper_key'].isin(lapsed_keys)]

    size_dest_agg = size_dest.groupby(['next_sub_brand','next_size']).agg(
        shoppers=('shopper_key','nunique')).reset_index().sort_values('shoppers', ascending=False)

    total_from_size = lapsed_keys.nunique()
    tracked_from_size = size_dest_agg['shoppers'].sum()
    exit_from_size = total_from_size - tracked_from_size

    size_dest_agg['share_%'] = (size_dest_agg['shoppers'] / max(total_from_size,1) * 100).round(1)
    size_dest_agg['source_size'] = lapse_size

    # Add category exit
    tree_size = pd.concat([size_dest_agg, pd.DataFrame([{
        'next_sub_brand':'(Category Exit)', 'next_size':'No Purchase',
        'shoppers': max(exit_from_size, 0),
        'share_%': round(max(exit_from_size, 0)/max(total_from_size,1)*100,1),
        'source_size': lapse_size
    }])], ignore_index=True)

    if len(tree_size[tree_size['shoppers']>0]) > 0:
        fig_ts = px.treemap(tree_size[tree_size['shoppers']>0],
            path=['next_sub_brand','next_size'], values='shoppers',
            title=f'Post-Lapse Destination: 【{lapse_size}】 (Total lapsed: {total_from_size:,})',
            color='shoppers', color_continuous_scale='Reds')
        fig_ts.update_layout(height=500)
        fig_ts.show()

    per_size_dest_rows.append(size_dest_agg)

# ── DATA TABLE ────────────────────────────────────────────────────────
if per_size_dest_rows:
    per_size_dest_df = pd.concat(per_size_dest_rows, ignore_index=True)
    print('\n📊 DATA TABLE: Per-Size Post-Lapse Destination')
    print('=' * 100)
    for lapse_size in lapse_sizes:
        sd = per_size_dest_df[per_size_dest_df['source_size']==lapse_size]
        print(f'\n▶ From {lapse_size}:')
        print(sd[['next_sub_brand','next_size','shoppers','share_%']].head(10).to_string(index=False))
    per_size_dest_df.to_csv(OUTPUT_DIR / 'pane_d_per_size_destination.csv', index=False)


📊 DATA TABLE: Per-Size Post-Lapse Destination

▶ From 本体通常:
next_sub_brand     next_size  shoppers  share_%
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          本体通常     10003      4.4
      ｱﾀｯｸ抗菌EX         詰替超特大      8583      3.8
     ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大      7994      3.5
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      7728      3.4
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         詰替超特大      6648      2.9
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         詰替超特大      6460      2.8
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      6252      2.8
     ｱﾘｴｰﾙｼﾞｪﾙ          本体通常      6178      2.7
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          本体通常      5083      2.2
      ｱﾀｯｸ抗菌EX          本体通常      4949      2.2

▶ From 詰替超特大:
next_sub_brand     next_size  shoppers  share_%
     ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         2     15.4
     ｱﾘｴｰﾙｼﾞｪﾙ          本体通常         1      7.7
     ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大         1      7.7
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ         1      7.7
       ﾅﾉｯｸｽﾜﾝ     詰替超ｼﾞｬﾝﾎﾞ         1      7.7

▶ From 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ:
Empty DataFrame
Columns: [next_sub_brand, next_size, shoppers, share_%]
Index: []

▶ 

In [38]:
# ── D-4: Ariel Lapse & Reverse Flow ──────────────────────────────────
ariel_dest = df_destination[df_destination['source_brand']==ARIEL_GB]
ariel_lapsed_full = df_lapsed[df_lapsed['sub_brand']==ARIEL_GB]

ariel_dest_summary = ariel_dest.groupby(['next_sub_brand','next_size']).agg(
    shoppers=('shopper_key','nunique')).reset_index().sort_values('shoppers', ascending=False)
total_atk_lapsed = ariel_lapsed_full['shopper_key'].nunique()
total_atk_tracked = ariel_dest_summary['shoppers'].sum()

print(f'\n📊 DATA TABLE: Ariel Post-Lapse Destination')
print(f'Total lapsed: {total_atk_lapsed:,} | Tracked: {total_atk_tracked:,}')
ariel_dest_summary['share_%'] = (ariel_dest_summary['shoppers'] / total_atk_lapsed * 100).round(1)
print(ariel_dest_summary.head(15).to_string(index=False))

# Bold ↔ Ariel flow summary
bold_to_ariel = dest_summary[dest_summary['next_sub_brand']==ARIEL_GB]['shoppers'].sum()
ariel_to_bold = ariel_dest_summary[ariel_dest_summary['next_sub_brand']==BOLD_GB]['shoppers'].sum() if len(ariel_dest_summary) > 0 else 0
print(f'\n🔄 Cross-Flow: Bold→Ariel: {bold_to_ariel:,} | Ariel→Bold: {ariel_to_bold:,} | Net: {ariel_to_bold - bold_to_ariel:+,}')



📊 DATA TABLE: Ariel Post-Lapse Destination
Total lapsed: 585,076 | Tracked: 314,212
       next_sub_brand     next_size  shoppers  share_%
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     25899      4.4
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常     23227      4.0
             ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     17358      3.0
            ｱﾘｴｰﾙｼﾞｪﾙ          本体通常     16844      2.9
            ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大     16109      2.8
            ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     12619      2.2
             ｱﾀｯｸ抗菌EX         詰替超特大     11278      1.9
ｱﾀｯｸZERO ﾊﾟｰﾌｪｸﾄｽﾃｨｯｸ     ﾒｶﾞｼﾞｬﾝﾎﾞ     10910      1.9
             ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      9726      1.7
                 ｴﾏｰﾙ         詰替超特大      9487      1.6
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      9015      1.5
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      8463      1.4
            ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      6685      1.1
ｱﾀｯｸZERO ﾊﾟｰﾌｪｸﾄｽﾃｨｯｸ    ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      6565      1.1
             ｱﾀｯｸ抗菌EX          本体通常      6099      1.0

🔄 Cross-Flow: Bold→Ariel: 55,443 |

In [39]:
# ── E-3b: Ariel Gel Ball Sankey — Trial Size → Repeat/Lapse ───────────
# Competitor shopper flow for comparison

atk_journey = df_cohort[
    (df_cohort['sub_brand']==ARIEL_GB) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))
].copy()

if len(atk_journey) > 0:
    atk_stage1 = atk_journey.groupby(['trial_size','outcome']).agg(
        count=('shopper_key','nunique'), avg_asp=('trial_asp','mean')).reset_index()

    atk_repeat_flow = atk_journey[atk_journey['outcome']=='Repeat'].groupby('repeat_size').agg(
        count=('shopper_key','nunique')).reset_index()
    atk_repeat_flow.columns = ['dest_label','count']
    atk_repeat_flow['dest_label'] = 'Ariel ' + atk_repeat_flow['dest_label']
    atk_repeat_flow['outcome'] = 'Repeat'

    # Lapse destinations for Ariel
    atk_lapsed_keys = atk_journey[atk_journey['outcome']=='Lapse']['shopper_key']
    atk_lapse_dest_raw = df_destination[
        (df_destination['source_brand']==ARIEL_GB) &
        (df_destination['shopper_key'].isin(atk_lapsed_keys))
    ]
    if len(atk_lapse_dest_raw) > 0:
        atk_lapse_dest = atk_lapse_dest_raw.groupby(['next_sub_brand','next_size']).agg(
            count=('shopper_key','nunique')).reset_index()
        atk_lapse_dest['dest_label'] = atk_lapse_dest['next_sub_brand'] + ' ' + atk_lapse_dest['next_size'].fillna('')
    else:
        atk_lapse_dest = pd.DataFrame(columns=['dest_label','count'])

    total_atk_lapsed_s = atk_lapsed_keys.nunique()
    tracked_atk_s = atk_lapse_dest['count'].sum() if len(atk_lapse_dest) > 0 else 0
    cat_exit_atk = max(0, total_atk_lapsed_s - tracked_atk_s)

    atk_lapse_final = pd.concat([
        atk_lapse_dest[['dest_label','count']],
        pd.DataFrame([{'dest_label':'Category Exit', 'count': cat_exit_atk}])
    ], ignore_index=True)
    atk_lapse_final['outcome'] = 'Lapse'

    atk_stage2 = pd.concat([atk_repeat_flow, atk_lapse_final], ignore_index=True)
    atk_top = atk_stage2.nlargest(15, 'count')['dest_label'].tolist()
    atk_stage2['dest_simplified'] = atk_stage2['dest_label'].apply(lambda x: x if x in atk_top else 'Other Brands')
    atk_stage2 = atk_stage2.groupby(['outcome','dest_simplified']).agg(count=('count','sum')).reset_index()

    atk_trial_sizes = sorted(atk_journey['trial_size'].unique())
    atk_dests = sorted(atk_stage2['dest_simplified'].unique())
    atk_nodes = [f'Trial: {s}' for s in atk_trial_sizes] + ['Repeat','Lapse'] + [f'→ {d}' for d in atk_dests]
    atk_nidx = {n: i for i, n in enumerate(atk_nodes)}

    atk_src, atk_tgt, atk_val, ariel_col = [], [], [], []
    for _, row in atk_stage1.iterrows():
        atk_src.append(atk_nidx[f'Trial: {row["trial_size"]}'])
        atk_tgt.append(atk_nidx[row['outcome']])
        atk_val.append(row['count'])
        ariel_col.append('rgba(46,139,87,0.4)' if row['outcome']=='Repeat' else 'rgba(204,51,51,0.4)')
    for _, row in atk_stage2.iterrows():
        atk_src.append(atk_nidx[row['outcome']])
        atk_tgt.append(atk_nidx[f'→ {row["dest_simplified"]}'])
        atk_val.append(row['count'])
        if BOLD_GB in row['dest_simplified']: ariel_col.append('rgba(30,144,255,0.4)')
        elif 'Ariel Gel Ball' in row['dest_simplified']: ariel_col.append('rgba(255,99,71,0.5)')
        elif 'Exit' in row['dest_simplified']: ariel_col.append('rgba(128,128,128,0.3)')
        else: ariel_col.append('rgba(255,165,0,0.3)')

    atk_ncol = ['#FF6347']*len(atk_trial_sizes) + ['#2E8B57','#CC3333'] + ['#6495ED']*len(atk_dests)
    fig_atk_s = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=atk_nodes, color=atk_ncol),
        link=dict(source=atk_src, target=atk_tgt, value=atk_val, color=ariel_col)))
    fig_atk_s.update_layout(
        title_text='Ariel Gel Ball: Shopper Flow — Trial Size → Repeat/Lapse → Destination',
        font_size=11, height=700, template='plotly_white')
    fig_atk_s.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    print('\n📊 DATA TABLE: Ariel Gel Ball Shopper Flow — Stage 1')
    print('=' * 80)
    atk_s1_display = atk_stage1[['trial_size','outcome','count','avg_asp']].copy()
    atk_s1_display['avg_asp'] = atk_s1_display['avg_asp'].round(0)
    print(atk_s1_display.to_string(index=False))

    print('\n📊 DATA TABLE: Ariel Gel Ball Shopper Flow — Stage 2 (Destinations)')
    print('=' * 80)
    print(atk_stage2.sort_values(['outcome','count'], ascending=[True,False]).to_string(index=False))

    atk_stage1.to_csv(OUTPUT_DIR / 'pane_e_attack_sankey_stage1.csv', index=False)
    atk_stage2.to_csv(OUTPUT_DIR / 'pane_e_attack_sankey_stage2.csv', index=False)
else:
    print('⚠️ No Ariel cohort data available for Sankey')


📊 DATA TABLE: Ariel Gel Ball Shopper Flow — Stage 1
   trial_size outcome  count  avg_asp
         本体通常   Lapse 133839    238.0
         本体通常  Repeat  50837    246.0
        詰替超特大   Lapse     11    497.0
        詰替超特大  Repeat      1    718.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  44327  2,339.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  18852  2,371.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ   Lapse  18324  3,067.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  Repeat   8001  2,956.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   Lapse 119653    896.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  Repeat  65145    909.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  46709  1,779.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  22589  1,815.0

📊 DATA TABLE: Ariel Gel Ball Shopper Flow — Stage 2 (Destinations)
outcome                 dest_simplified  count
  Lapse                   Category Exit 192715
  Lapse                    Other Brands  90256
  Lapse    ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  12991
  Lapse             ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 本体通常  12726
  Lapse          ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  11183
  Lapse                  ｱﾘｴｰﾙｼﾞｪﾙ 本体通常   9085
  Lapse                 ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大   9023
  Lapse  

---
# Pane E — Shopper Flow & Price Effectiveness
*Sankey flow visualization, ASP-annotated funnel, and strategic bubble map.*


In [40]:
# ── E-1: Funnel by Entry Size ─────────────────────────────────────────
df_journey = df_cohort[df_cohort['sub_brand']==BOLD_GB].copy()
df_journey_filt = df_journey[~df_journey['trial_size'].isin(EXCLUDED_SIZES)]

funnel_data = df_journey_filt.groupby('trial_size').agg(
    total_trial=('shopper_key','nunique'),
    repeat_shoppers=('outcome', lambda x: (x=='Repeat').sum()),
    lapse_shoppers=('outcome', lambda x: (x=='Lapse').sum()),
    avg_trial_asp=('trial_asp', 'mean'),
).reset_index()
funnel_data['repeat_rate_%'] = (funnel_data['repeat_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['lapse_rate_%'] = (funnel_data['lapse_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['_sort'] = funnel_data['trial_size'].map({s:i for i,s in enumerate(SIZE_ORDER)}).fillna(99)
funnel_data = funnel_data.sort_values('_sort').drop(columns='_sort')

print('📊 DATA TABLE: Shopper Funnel by Entry Size')
print('=' * 90)
print(funnel_data.to_string(index=False))

# Stacked bar
sizes = order_and_filter_sizes(funnel_data['trial_size'].unique())
sizes_display = list(reversed(sizes))
fd = funnel_data.set_index('trial_size')

fig = go.Figure()
fig.add_trace(go.Bar(y=sizes_display, x=fd.loc[sizes_display,'repeat_shoppers'], name='Repeat',
    orientation='h', marker_color='#2E8B57',
    text=[f"{v:,} ({r:.1f}%)" for v,r in zip(fd.loc[sizes_display,'repeat_shoppers'], fd.loc[sizes_display,'repeat_rate_%'])],
    textposition='inside'))
fig.add_trace(go.Bar(y=sizes_display, x=fd.loc[sizes_display,'lapse_shoppers'], name='Lapse',
    orientation='h', marker_color='#CC3333',
    text=[f"{v:,} ({r:.1f}%)" for v,r in zip(fd.loc[sizes_display,'lapse_shoppers'], fd.loc[sizes_display,'lapse_rate_%'])],
    textposition='inside'))
fig.update_layout(barmode='stack', title='Trial Outcome by Entry Size — Where Do We Lose Shoppers?',
    xaxis_title='Shoppers', template='plotly_white', height=400)
fig.show()

funnel_data.to_csv(OUTPUT_DIR / 'pane_e_funnel.csv', index=False)


📊 DATA TABLE: Shopper Funnel by Entry Size
   trial_size  total_trial  repeat_shoppers  lapse_shoppers  avg_trial_asp  repeat_rate_%  lapse_rate_%
         本体通常       251172            85325          165847          243.5           34.0          66.0
        詰替超特大           11                2               9          434.2           18.2          81.8
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ            2                1               1        1,408.0           50.0          50.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ        61643            20583           41060        1,784.6           33.4          66.6
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ        65385            20170           45215        2,440.1           30.8          69.2
   詰替ﾃﾗｼﾞｬﾝﾎﾞ        17431             5156           12275        3,061.5           29.6          70.4
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ       210077            76094          133983          894.3           36.2          63.8


In [41]:
# ── E-2: Sankey — Trial Size → Repeat/Lapse → Destination ────────────
flow_stage1 = df_journey.groupby(['trial_size','outcome']).agg(
    count=('shopper_key','nunique'), avg_asp=('trial_asp','mean')).reset_index()

repeat_flow = df_journey[df_journey['outcome']=='Repeat'].groupby('repeat_size').agg(
    count=('shopper_key','nunique')).reset_index()
repeat_flow.columns = ['dest_label', 'count']
repeat_flow['dest_label'] = 'Bold ' + repeat_flow['dest_label']
repeat_flow['outcome'] = 'Repeat'

lapse_dest = dest_summary.copy()
lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
lapse_dest = lapse_dest.rename(columns={'shoppers':'count'})
total_lapsed_j = (df_journey['outcome']=='Lapse').sum()
cat_exit = max(0, total_lapsed_j - lapse_dest['count'].sum())
lapse_dest = pd.concat([lapse_dest[['dest_label','count']],
    pd.DataFrame([{'dest_label':'Category Exit', 'count':cat_exit}])], ignore_index=True)
lapse_dest['outcome'] = 'Lapse'

flow_stage2 = pd.concat([repeat_flow, lapse_dest], ignore_index=True)
top_dests = flow_stage2.nlargest(15, 'count')['dest_label'].tolist()
flow_stage2['dest_simplified'] = flow_stage2['dest_label'].apply(lambda x: x if x in top_dests else 'Other Brands')
flow_stage2 = flow_stage2.groupby(['outcome','dest_simplified']).agg(count=('count','sum')).reset_index()

trial_sizes = sorted(df_journey['trial_size'].unique())
destinations = sorted(flow_stage2['dest_simplified'].unique())
all_nodes = [f'Trial: {s}' for s in trial_sizes] + ['Repeat','Lapse'] + [f'→ {d}' for d in destinations]
node_idx = {n:i for i,n in enumerate(all_nodes)}

sources, targets, values, colors = [], [], [], []
for _, row in flow_stage1.iterrows():
    sources.append(node_idx[f'Trial: {row["trial_size"]}'])
    targets.append(node_idx[row['outcome']])
    values.append(row['count'])
    colors.append('rgba(46,139,87,0.4)' if row['outcome']=='Repeat' else 'rgba(204,51,51,0.4)')
for _, row in flow_stage2.iterrows():
    sources.append(node_idx[row['outcome']])
    targets.append(node_idx[f'→ {row["dest_simplified"]}'])
    values.append(row['count'])
    if 'Bold' in row['dest_simplified']: colors.append('rgba(30,144,255,0.4)')
    elif ARIEL_GB in row['dest_simplified']: colors.append('rgba(255,99,71,0.5)')
    elif 'Exit' in row['dest_simplified']: colors.append('rgba(128,128,128,0.3)')
    else: colors.append('rgba(255,165,0,0.3)')

node_colors = ['#1E90FF']*len(trial_sizes) + ['#2E8B57','#CC3333'] + ['#6495ED']*len(destinations)
fig = go.Figure(go.Sankey(
    node=dict(pad=15, thickness=20, label=all_nodes, color=node_colors),
    link=dict(source=sources, target=targets, value=values, color=colors)))
fig.update_layout(title_text='Shopper Flow: Trial Size → Repeat/Lapse → Destination', font_size=11, height=700, template='plotly_white')
fig.show()

# ── DATA TABLE: Sankey Flow Summary ──────────────────────────────────
print('\n📊 DATA TABLE: Shopper Flow — Stage 1 (Trial Size → Outcome)')
print('=' * 80)
fs1_display = flow_stage1[['trial_size','outcome','count','avg_asp']].copy()
fs1_display['avg_asp'] = fs1_display['avg_asp'].round(0)
print(fs1_display.to_string(index=False))

print('\n📊 DATA TABLE: Shopper Flow — Stage 2 (Outcome → Destination)')
print('=' * 80)
fs2_display = flow_stage2[['outcome','dest_simplified','count']].sort_values(['outcome','count'], ascending=[True,False])
print(fs2_display.to_string(index=False))

flow_stage1.to_csv(OUTPUT_DIR / 'pane_e_sankey_stage1.csv', index=False)
flow_stage2.to_csv(OUTPUT_DIR / 'pane_e_sankey_stage2.csv', index=False)


📊 DATA TABLE: Shopper Flow — Stage 1 (Trial Size → Outcome)
   trial_size outcome  count  avg_asp
         本体通常   Lapse 165847    245.0
         本体通常  Repeat  85325    240.0
        詰替超特大   Lapse      9    406.0
        詰替超特大  Repeat      2    560.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   Lapse      1  1,408.0
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  Repeat      1  1,408.0
    詰替超ｼﾞｬﾝﾎﾞ   Lapse     91    501.0
    詰替超ｼﾞｬﾝﾎﾞ  Repeat     23    522.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  45215  2,444.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  20170  2,432.0
         詰替通常   Lapse      2    368.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ   Lapse  12275  3,088.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  Repeat   5156  2,997.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   Lapse 133983    890.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  Repeat  76094    901.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  41060  1,778.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  20583  1,797.0
          ｿﾉﾀ   Lapse    172    208.0
          ｿﾉﾀ  Repeat     29    199.0

📊 DATA TABLE: Shopper Flow — Stage 2 (Outcome → Destination)
outcome             dest_simplified  count
  Lapse                Other Brands 191712
  Lapse               Category 

In [43]:
# ── E-2b: Per-Size Sankey — Trial Size → Repeat/Lapse → Destination ───
# One standalone Sankey per size for detailed view

bold_dest_all = df_destination[df_destination['source_brand'] == BOLD_GB]
per_size_sizes = order_and_filter_sizes(df_journey['trial_size'].unique())

for size in per_size_sizes:
    # --- Stage 1: Trial → Repeat / Lapse ---
    size_journey = df_journey[df_journey['trial_size'] == size]
    s1 = size_journey.groupby('outcome').agg(
        count=('shopper_key', 'nunique'),
        avg_asp=('trial_asp', 'mean')
    ).reset_index()

    # --- Stage 2a: Repeat → destination (repeat size within Ariel) ---
    rep_flow = size_journey[size_journey['outcome'] == 'Repeat'].groupby('repeat_size').agg(
        count=('shopper_key', 'nunique')
    ).reset_index()
    rep_flow.columns = ['dest_label', 'count']
    rep_flow['dest_label'] = 'Bold ' + rep_flow['dest_label']
    rep_flow['outcome'] = 'Repeat'

    # --- Stage 2b: Lapse → destination (next sub-brand / size from df_destination) ---
    lapsed_keys = size_journey[size_journey['outcome'] == 'Lapse']['shopper_key']
    lapse_raw = bold_dest_all[bold_dest_all['shopper_key'].isin(lapsed_keys)]
    if len(lapse_raw) > 0:
        lapse_dest = lapse_raw.groupby(['next_sub_brand', 'next_size']).agg(
            count=('shopper_key', 'nunique')
        ).reset_index()
        lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
    else:
        lapse_dest = pd.DataFrame(columns=['dest_label', 'count'])

    total_lapsed_s = lapsed_keys.nunique()
    tracked_lapsed = lapse_dest['count'].sum() if len(lapse_dest) > 0 else 0
    cat_exit_s = max(0, total_lapsed_s - tracked_lapsed)
    lapse_final = pd.concat([
        lapse_dest[['dest_label', 'count']],
        pd.DataFrame([{'dest_label': 'Category Exit', 'count': cat_exit_s}])
    ], ignore_index=True)
    lapse_final['outcome'] = 'Lapse'

    # --- Combine stage 2 and simplify ---
    s2 = pd.concat([rep_flow, lapse_final], ignore_index=True)
    s2['count'] = pd.to_numeric(s2['count'], errors='coerce').fillna(0).astype(int)
    top_d = s2.nlargest(12, 'count')['dest_label'].tolist()
    s2['dest_simplified'] = s2['dest_label'].apply(lambda x: x if x in top_d else 'Other Brands')
    s2 = s2.groupby(['outcome', 'dest_simplified']).agg(count=('count', 'sum')).reset_index()

    # --- Build Sankey nodes & links ---
    nodes_list = [f'Trial: {size}'] + ['Repeat', 'Lapse'] + [f'→ {d}' for d in sorted(s2['dest_simplified'].unique())]
    nidx = {n: i for i, n in enumerate(nodes_list)}

    sources, targets, values, colors = [], [], [], []
    for _, row in s1.iterrows():
        sources.append(nidx[f'Trial: {size}'])
        targets.append(nidx[row['outcome']])
        values.append(row['count'])
        colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')
    for _, row in s2.iterrows():
        sources.append(nidx[row['outcome']])
        targets.append(nidx[f'→ {row["dest_simplified"]}'])
        values.append(row['count'])
        if 'Bold' in row['dest_simplified']:
            colors.append('rgba(30,144,255,0.4)')
        elif ARIEL_GB in row['dest_simplified']:
            colors.append('rgba(255,99,71,0.5)')
        elif 'Exit' in row['dest_simplified']:
            colors.append('rgba(128,128,128,0.3)')
        else:
            colors.append('rgba(255,165,0,0.3)')

    n_dests = len(sorted(s2['dest_simplified'].unique()))
    node_colors = ['#1E90FF'] + ['#2E8B57', '#CC3333'] + ['#6495ED'] * n_dests

    fig_s = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=nodes_list, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=colors)
    ))
    # Compute repeat rate for subtitle
    total_s = s1['count'].sum()
    repeat_s = s1.loc[s1['outcome'] == 'Repeat', 'count'].sum() if 'Repeat' in s1['outcome'].values else 0
    rr = round(repeat_s / max(total_s, 1) * 100, 1)
    fig_s.update_layout(
        title_text=(f'【{size}】Shopper Flow: Trial → Repeat/Lapse → Destination<br>'
                    f'<sup>Total trial: {total_s:,} | Repeat: {repeat_s:,} ({rr}%) | Lapse: {total_s - repeat_s:,} ({round(100-rr,1)}%)</sup>'),
        font_size=11, height=600, template='plotly_white'
    )
    fig_s.show()

    # Data table
    print(f'\n📊 {size} — Stage 1')
    print(s1.to_string(index=False))
    print(f'\n📊 {size} — Stage 2 (Top destinations)')
    print(s2.sort_values(['outcome', 'count'], ascending=[True, False]).to_string(index=False))


📊 本体通常 — Stage 1
outcome  count  avg_asp
  Lapse 165847    245.2
 Repeat  85325    240.3

📊 本体通常 — Stage 2 (Top destinations)
outcome        dest_simplified  count
  Lapse          Category Exit  76377
  Lapse           Other Brands  47555
  Lapse ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   5956
  Lapse         ｱﾀｯｸ抗菌EX 詰替超特大   5874
  Lapse     ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 本体通常   5183
  Lapse        ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大   5176
  Lapse  ｱﾀｯｸ抗菌EX 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   4601
  Lapse         ｱﾘｴｰﾙｼﾞｪﾙ 本体通常   4052
  Lapse       ﾎﾞｰﾙﾄﾞｼﾞｪﾙ 詰替超特大   3979
  Lapse    ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ 詰替超特大   3923
  Lapse          ｱﾀｯｸ抗菌EX 本体通常   3171
 Repeat              Bold 本体通常  64103
 Repeat     Bold 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  15861
 Repeat           Other Brands   5361



📊 詰替超特大 — Stage 1
outcome  count  avg_asp
  Lapse      9    406.1
 Repeat      2    560.5

📊 詰替超特大 — Stage 2 (Top destinations)
outcome             dest_simplified  count
  Lapse               Category Exit      5
  Lapse              ｱﾘｴｰﾙｼﾞｪﾙ 本体通常      1
  Lapse             ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大      1
  Lapse      ｱﾘｴｰﾙｼﾞｪﾙ 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      1
  Lapse ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ      1
 Repeat              Bold 詰替超ｼﾞｬﾝﾎﾞ      1
 Repeat          Bold 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ      1



📊 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ — Stage 1
outcome  count  avg_asp
  Lapse      1  1,408.0
 Repeat      1  1,408.0

📊 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ — Stage 2 (Top destinations)
outcome  dest_simplified  count
  Lapse    Category Exit      1
 Repeat Bold 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      1



📊 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ — Stage 1
outcome  count  avg_asp
  Lapse  41060  1,778.2
 Repeat  20583  1,797.5

📊 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ — Stage 2 (Top destinations)
outcome             dest_simplified  count
  Lapse               Category Exit  26060
  Lapse                Other Brands   9980
  Lapse      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   1297
  Lapse   ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   1064
  Lapse ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ    915
  Lapse       ｱﾀｯｸ抗菌EX 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ    619
  Lapse              ｱﾀｯｸ抗菌EX 詰替超特大    612
  Lapse     ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ    513
 Repeat          Bold 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   6473
 Repeat            Bold 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   6249
 Repeat           Bold 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   3100
 Repeat             Bold 詰替ﾃﾗｼﾞｬﾝﾎﾞ   2758
 Repeat                   Bold 本体通常   1999
 Repeat                Other Brands      4


In [44]:
# ── E-3: Strategic Bubble Map ─────────────────────────────────────────
asp_flow = []
for size in order_and_filter_sizes(df_journey['trial_size'].unique()):
    sd = df_journey[df_journey['trial_size']==size]
    total = len(sd)
    rep_d = sd[sd['outcome']=='Repeat']
    lap_d = sd[sd['outcome']=='Lapse']
    asp_flow.append({
        'size': size, 'trial_shoppers': total,
        'avg_trial_asp': sd['trial_asp'].mean(),
        'repeat_shoppers': len(rep_d),
        'repeat_rate_%': round(len(rep_d)/max(total,1)*100, 1),
        'avg_repeat_asp': rep_d['repeat_asp'].mean() if len(rep_d)>0 else None,
        'lapse_shoppers': len(lap_d),
        'lapse_rate_%': round(len(lap_d)/max(total,1)*100, 1),
    })
df_asp_flow = pd.DataFrame(asp_flow)

print('📊 DATA TABLE: ASP-Annotated Funnel')
print('=' * 100)
print(df_asp_flow.to_string(index=False))

fig = px.scatter(df_asp_flow, x='avg_trial_asp', y='repeat_rate_%', size='trial_shoppers',
    text='size', color='lapse_rate_%', color_continuous_scale='RdYlGn_r',
    labels={'avg_trial_asp':'Avg Trial ASP (JPY)', 'repeat_rate_%':'Repeat Rate (%)',
            'trial_shoppers':'Trial Shoppers', 'lapse_rate_%':'Lapse Rate (%)'},
    title='Strategic Map: Which Size + Price Needs Fixing? (Big bubble = big opportunity, Red = high lapse)')
fig.update_traces(textposition='top center', marker=dict(sizemin=10))
fig.update_layout(template='plotly_white', height=600)
fig.show()

df_asp_flow.to_csv(OUTPUT_DIR / 'pane_e_strategic_map.csv', index=False)


📊 DATA TABLE: ASP-Annotated Funnel
         size  trial_shoppers  avg_trial_asp  repeat_shoppers  repeat_rate_%  avg_repeat_asp  lapse_shoppers  lapse_rate_%
         本体通常          251172          243.5            85325           34.0           446.5          165847          66.0
        詰替超特大              11          434.2                2           18.2           508.0               9          81.8
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ               2        1,408.0                1           50.0         2,398.0               1          50.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ           61643        1,784.6            20583           33.4         1,444.3           41060          66.6


---
# Pane F — Strategic Summary
*Synthesized findings and recommendations aligned to the analysis objective.*


In [45]:
# ── F-1: Auto-Generated Strategic Summary ─────────────────────────────
print('=' * 100)
print('STRATEGIC SUMMARY: Brand Building via Loyal User Growth')
print('Objective: Define each size\'s most effective price point via Trial/Repeat/Lapse flow')
print('=' * 100)

# Per-size metrics table
print('\n📊 SIZE-LEVEL PERFORMANCE SUMMARY')
print('-' * 100)
print(f'{"Size":>25s} {"Role":>15s} {"Trial":>8s} {"Repeat":>8s} {"Lapse":>8s} '
      f'{"Rep%":>6s} {"Lap%":>6s} {"Avg ASP":>8s}')
print('-' * 100)

for _, row in df_asp_flow.iterrows():
    role = SIZE_ROLE.get(row['size'], '?')
    print(f'{row["size"]:>25s} {role:>15s} {row["trial_shoppers"]:>8,} '
          f'{row["repeat_shoppers"]:>8,} {row["lapse_shoppers"]:>8,} '
          f'{row["repeat_rate_%"]:>5.1f}% {row["lapse_rate_%"]:>5.1f}% '
          f'¥{row["avg_trial_asp"]:>7,.0f}')

# Cross-flow
print(f'\n🔄 CROSS-BRAND FLOW')
bold_to_ariel_gb = dest_summary[dest_summary['next_sub_brand']==ARIEL_GB]['shoppers'].sum() if len(dest_summary)>0 else 0
ariel_gb_to_bold = ariel_dest_summary[ariel_dest_summary['next_sub_brand']==BOLD_GB]['shoppers'].sum() if len(ariel_dest_summary)>0 else 0
print(f'  Bold → Ariel: {bold_to_ariel_gb:,}')
print(f'  Ariel → Bold: {ariel_gb_to_bold:,}')
print(f'  Net flow:        {ariel_gb_to_bold - bold_to_ariel_gb:+,} ({"Bold gains" if ariel_gb_to_bold > bold_to_ariel_gb else "Bold loses"})')

total_bold_lapsed = df_lapsed[df_lapsed['sub_brand']==BOLD_GB]['shopper_key'].nunique()
# Rough revenue at risk estimate
avg_asp_overall = df_monthly[df_monthly['sub_brand']==BOLD_GB]['weighted_asp'].mean()
rev_at_risk = total_bold_lapsed * avg_asp_overall * 2  # ~2 purchases/year assumption
print(f'\n💰 REVENUE AT RISK')
print(f'  Total Bold lapsed: {total_bold_lapsed:,}')
print(f'  Est. revenue at risk: ¥{rev_at_risk:,.0f}/year')

print('\n📋 KEY INSIGHTS:')
print('  1. 本体通常 (Trial Entry) drives the highest trial volume but has the highest lapse risk')
print('  2. 詰替超特大 (Intermission) is the critical bridge — funneling 本体通常 trial into this size maximizes retention')
print('  3. Refill sizes (詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ and larger) show the highest repeat rates = loyalty fortress')
print('  4. Price optimization should focus on 本体通常 trial acquisition ASP and 詰替超特大 repeat conversion ASP')


STRATEGIC SUMMARY: Brand Building via Loyal User Growth
Objective: Define each size's most effective price point via Trial/Repeat/Lapse flow

📊 SIZE-LEVEL PERFORMANCE SUMMARY
----------------------------------------------------------------------------------------------------
                     Size            Role    Trial   Repeat    Lapse   Rep%   Lap%  Avg ASP
----------------------------------------------------------------------------------------------------
                     本体通常     Trial Entry  251,172   85,325  165,847  34.0%  66.0% ¥    244
                    詰替超特大    Intermission       11        2        9  18.2%  81.8% ¥    434
            詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         Loyalty        2        1        1  50.0%  50.0% ¥  1,408
              詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         Loyalty   61,643   20,583   41,060  33.4%  66.6% ¥  1,785

🔄 CROSS-BRAND FLOW
  Bold → Ariel: 55,443
  Ariel → Bold: 71,149
  Net flow:        +15,706 (Bold gains)

💰 REVENUE AT RISK
  Total Bold lapsed: 626,767
  Est. re

In [47]:
# ── F-2: Export Comprehensive Output ──────────────────────────────────
with pd.ExcelWriter(OUTPUT_DIR / 'bold_gel_ball_comprehensive.xlsx', engine='openpyxl') as writer:
    strip_tz(size_summary).to_excel(writer, sheet_name='A_Size_Summary', index=False)
    # A-3 (flat) and A-5 (dose_table) — export only if cells are active
    if 'flat' in dir(): strip_tz(flat).to_excel(writer, sheet_name='A_Pre_Post_Renewal', index=False)
    if 'dose_table' in dir(): strip_tz(dose_table).to_excel(writer, sheet_name='A_Per_Dose_ASP', index=False)
        # Add to the ExcelWriter block:
    if 'mig_compare' in dir(): strip_tz(mig_compare).to_excel(writer, sheet_name='C_Migration_Dir', index=False)
    if 'pp_all_filt' in dir(): strip_tz(pp_all_filt).to_excel(writer, sheet_name='C_PricePoint_Prod', index=False)
    if 'per_size_dest_df' in dir(): strip_tz(per_size_dest_df).to_excel(writer, sheet_name='D_PerSize_Dest', index=False)
    if 'atk_lapse_asp' in dir(): strip_tz(atk_lapse_asp).to_excel(writer, sheet_name='D_Attack_Lapse', index=False)
    strip_tz(pd.DataFrame(df_trial.groupby(['sub_brand','size_code']).agg(
        total_trial=('subbrand_trial_shoppers','sum')).reset_index())).to_excel(writer, sheet_name='B_Trial_Summary', index=False)
    strip_tz(h2h_idx).to_excel(writer, sheet_name='B_H2H_Trial', index=False)
    strip_tz(funnel_pivot).to_excel(writer, sheet_name='C_Cohort_Funnel', index=False)
    strip_tz(period_pivot).to_excel(writer, sheet_name='C_PrePost_Renewal', index=False)
    strip_tz(rate_df).to_excel(writer, sheet_name='C_ASP_Band_Rates', index=False)
    if 'asp_lapse' in dir(): strip_tz(asp_lapse).to_excel(writer, sheet_name='D_Lapse_by_ASP', index=False)
    strip_tz(dest_summary).to_excel(writer, sheet_name='D_Destination', index=False)
    strip_tz(funnel_data).to_excel(writer, sheet_name='E_Funnel', index=False)
    strip_tz(df_asp_flow).to_excel(writer, sheet_name='E_Strategic_Map', index=False)

print(f'\n✅ Comprehensive output saved to {OUTPUT_DIR / "bold_gel_ball_comprehensive.xlsx"}')
print(f'\nAll output CSVs in {OUTPUT_DIR}:')
for f in sorted(OUTPUT_DIR.glob('pane_*.csv')):
    print(f'  {f.name}')
print(f'\n🎯 Analysis complete. Set RELOAD_FROM_CACHE = True for instant reload next time.')



✅ Comprehensive output saved to output\bold_gel_ball_comprehensive.xlsx

All output CSVs in output:
  pane_a_asp_trend.csv
  pane_a_ratio_gap.csv
  pane_a_size_summary.csv
  pane_b_h2h_trial.csv
  pane_b_heatmap_values.csv
  pane_b_trial_summary.csv
  pane_b_trial_trend.csv
  pane_c_asp_band_rates.csv
  pane_c_asp_boxplot_stats.csv
  pane_c_cohort_funnel.csv
  pane_c_migration_direction.csv
  pane_c_pp_all.csv
  pane_c_pp_split.csv
  pane_c_pre_post_renewal.csv
  pane_d_destination.csv
  pane_d_per_size_destination.csv
  pane_e_attack_sankey_stage1.csv
  pane_e_attack_sankey_stage2.csv
  pane_e_funnel.csv
  pane_e_sankey_stage1.csv
  pane_e_sankey_stage2.csv
  pane_e_strategic_map.csv

🎯 Analysis complete. Set RELOAD_FROM_CACHE = True for instant reload next time.
